# Path definition

In [1]:
## Please define here the working path

pathwork = "E:/Work/DIAMOND/NeW/"

# Structurte of the folders:
# Main
#  ->
#  ->


# Retrievement and calculation of the full EXIOBASE with pymrio

In [2]:
# To run only once, the first time

In [3]:
# # Load libraries
# import pandas as pd
# import pymrio

# # Define working path
# path_exiodata = pathwork + "data_raw/EXIOBASE/"
# path_data = pathwork + "data_raw/"

# # If not yet downloaded
# #exiodata_downloadlog = pymrio.download_exiobase3(storage_folder=path_exiodata, system="ixi", years=[2019])


In [4]:
# # Define year to load
# LOADYEAR = "2019Y1"

# # Path + file name
# path_exio = pathwork + f"data_raw/EXIOBASE/IOT_{LOADYEAR[:-2]}_ixi.zip"

# # Parse raw data with pymrio
# exio19 = pymrio.parse_exiobase3(path=path_exio)

# exio19.get_regions()

# # Save data
# save_folder_full = pathwork + "data_raw/EXIOBASE/pymrio/"
# exio19.calc_all()
# exio19.save_all(path=save_folder_full)

### Import full Exiobase for 2019

In [5]:
# To run once EXIOBASE are loaded and full database calculated
# NB: EXIOBASE data is not on the Github folder, for size limitation reason 

import pymrio
import os
import iode as io
import numpy as np
import pandas as pd
import re

# Load previously extracted and saved 
save_folder_full = pathwork + "data_raw/EXIOBASE/pymrio/"
exio19 = pymrio.load_all(path=save_folder_full)

# Initial list and values
BASEYEAR = "2019Y1"
LOADYEAR = "2019Y1"
SMPSTRY = "2015Y1"
SMPENDY = "2050Y1"
SMP = SMPSTRY + ":" + SMPENDY
print(SMP)
io.variables.sample = f'{SMP}'

2015Y1:2050Y1


### Show database content

In [6]:
#print(exio19)
#print(exio19.impacts.D_imp)
#print(exio19.satellite.D_imp)
#print(exio19.x)
#print(exio19.Y)
#print(exio19.impacts.F)
#print(exio19.satellite.F)
#exio19.get_regions()
#exio19.get_sectors()
#exio19.get_Y_categories()
#print(exio19.Z.sum().sum())
#print(exio19.Y.sum().sum())
#exio19.impacts.F
#print(exio19.F.columns.get_level_values("region").unique())

### Aggregation of initial database

In [7]:
# Caution does not work when running the code several times, mrio base must be reloaded
# Modify codes lists

## Rename final demands indexes
lst_ini_FD = exio19.get_Y_categories()
# Final consumption expenditure by households: CONSHV
# Final consumption expenditure by non-profit organisations serving households (NPISH): CONSNPV
# Final consumption expenditure by government: CONSGV
# Gross fixed capital formation: DINVV
# Changes in inventories: DSTOCKV
# Changes in valuables: DVALUEV
# Exports: Total (fob): EXPTOTV
lst_upd_FD = ["CONSHV","CONSNPV","CONSGV","DINVV","DSTOCKV","DVALUEV","EXPTOTV"]
dict_FD = dict(zip(lst_ini_FD, lst_upd_FD))
exio19.rename_Y_categories(dict_FD)

## Rename sectors only with numbers (3-digit)
# Do a list of numbers from 1 to 164 with zero(s) before
lst_upd_Sect = [str(i).zfill(3) for i in range(1, 164)]
lst_ini_Sect = exio19.get_sectors()
dict_Sect = dict(zip(lst_ini_Sect, lst_upd_Sect))
exio19.rename_sectors(dict_Sect)

## Mapping of the Sectors
map_sect = pd.read_excel(os.path.join(save_folder_full,"Mapping_Sectors.xlsx"), dtype=str) # Excel file containing sectors' mapping for aggregation
map_ini_sect = map_sect["Sect_cd_ini"]          # Initial sectors
map_upd_sect = map_sect["Sect_cd_final"]        # Aggregated sectors
dict_map_sect = dict(zip(map_ini_sect, map_upd_sect)) 
exio19.rename_sectors(dict_map_sect)            # Replace names
exio19.aggregate_duplicates()                   # Groupby

lst_sec = map_sect["Sect_cd_final"].unique()

## Mapping of the regions
map_reg = pd.read_excel(os.path.join(save_folder_full,"Mapping_Regions.xlsx"), dtype=str) # Excel file containing regions' mapping for aggregation
map_ini_reg = map_reg["Reg_cd_ini"]             # Initial regions
map_upd_reg = map_reg["Reg_cd_final"]           # Aggregated regions
dict_map_reg = dict(zip(map_ini_reg, map_upd_reg))
exio19.rename_regions(dict_map_reg)             # Replace names
exio19.aggregate_duplicates()                   # Groupby

lst_reg = map_reg["Reg_cd_final"].unique()


c:\Users\bapti\AppData\Local\Programs\Python\Python312\Lib\site-packages\pymrio\core\mriosystem.py:2177: FutureWarning: The 'axis' keyword in DataFrame.groupby is deprecated and will be removed in a future version.
  df.groupby(df.index, axis=0, sort=False)
c:\Users\bapti\AppData\Local\Programs\Python\Python312\Lib\site-packages\pymrio\core\mriosystem.py:2179: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  .groupby(df.columns, axis=1, sort=False)
c:\Users\bapti\AppData\Local\Programs\Python\Python312\Lib\site-packages\pymrio\core\mriosystem.py:2177: FutureWarning: The 'axis' keyword in DataFrame.groupby is deprecated and will be removed in a future version.
  df.groupby(df.index, axis=0, sort=False)
c:\Users\bapti\AppData\Local\Programs\Python\Python312\Lib\site-packages\pymrio\core\mriosystem.py:2179: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  .groupby(df.c

#### Loading calculated GFCF matrix

In [8]:
# Define working paths
path_mat_inv = pathwork + "/Data_Raw/GFCF/"
# Load GFCF Intensity to allocate total GFCF to sector (GFCF intensity have been calculated previously, used here as input) 
data_mat_inv = os.path.join(path_mat_inv, "Inv_mat_balanced.csv")
mat_gfcf = pd.read_csv(data_mat_inv, sep=";", dtype={1: str, 0: str}, header=[0, 1], index_col=[0, 1])
# to transform index value 1,2,3 etc; into 01,02,03 etc.
mat_gfcf.index = mat_gfcf.index.set_levels(mat_gfcf.index.levels[1].map(lambda x: f"{int(x):02}"), level=1)
# Calculation of DINVV
mat_gfcf_dinv = mat_gfcf.sum(axis=0)
# Calculation of ADDDINVV
mat_gfcf_addinv = mat_gfcf.T.groupby("col_reg").sum()
#print(mat_gfcf_addinv.head())
print(mat_gfcf_addinv.head())
print(mat_gfcf_dinv.head())

row_reg          AU                                                        \
row_sec          01            02            03            04          05   
col_reg                                                                     
AU       836.981406  5.226820e+02  3.856703e+01  9.785099e-01  450.611802   
BR         0.121949  1.166712e-02  2.357401e-06  1.457761e-09    0.000007   
CA         0.000003  2.800691e-06  4.836812e-07  1.477110e-07    0.000010   
CN         0.000008  4.965779e-09  3.803432e-06  1.000001e-08    0.000010   
DE         0.023227  7.305909e-03  8.855344e-03  1.679005e-03    0.276881   

row_reg                                                                  ...  \
row_sec          06           07           08          09            10  ...   
col_reg                                                                  ...   
AU       327.370051  2500.182331  9897.902803  121.825838  3.998578e+02  ...   
BR         0.000016     0.000058     0.029191    0.000003  2.99

### Load scalars to calculate KSTOCK (based on average sectoral capital stock intensity, over production)

In [9]:
## Load scalars of intensity
# Define working paths
path_kstock = pathwork + "data_raw/KSTOCK/"
# Load GFCF intensity to allocate total GFCF to sector 
scl_kstock = os.path.join(path_kstock, "Scl_int_kstock.csv")
int_kstock = pd.read_csv(scl_kstock , sep=";", header=[0], dtype={"Sec_cd": str})


### Main variables calculation from EXIOBASE

In [10]:
dict_var = dict() # Dictionnay that will store all variables
val = {f"{year}Y1": np.nan for year in range(int(SMPSTRY[:-2]), int(SMPENDY[:-2])+1)} # dictionnay to store variables' value according to pre-define sample

####################################################
# Final demands
##### DINVV
# Use GFCF calculated previously
# Store values into the dictionnary
for rd in mat_gfcf_dinv.index.get_level_values("col_reg").unique():
    for sd in mat_gfcf_dinv.index.get_level_values("col_sec").unique():
        val[f"{BASEYEAR}"] = mat_gfcf_dinv.loc[(rd,sd),]
        dict_var[f"DINVV_{rd}_{sd}"] = {"val": val.copy(), "cmt": f"Gross fixed capital formation, nominal", "unit": "current million euros", "idt": "", "eqs": ""}

# To verify that sum of all ADDINVV equals DINVV
print(exio19.Y.loc[:,(exio19.Y.columns.get_level_values("category") == "DINVV")].sum().T.sum(), mat_gfcf_dinv.sum())

##### DSTOCKV and DVALUEV
# Caution summation of final demands : DSTOCKV and DVALUEV in one DSTOCKV
DSTOCKV = exio19.Y.loc[:,(exio19.Y.columns.get_level_values("category") == "DSTOCKV")].groupby(level=1).sum()
DVALUEV = exio19.Y.loc[:,(exio19.Y.columns.get_level_values("category") == "DVALUEV")].groupby(level=1).sum()
# Concatenation of both dataframe
DSTOCKV_sum = pd.concat([DSTOCKV, DVALUEV], axis=1)
# Sum for each region in columns both final demands 
DSTOCKV = DSTOCKV_sum.T.groupby("region", sort=False).sum().T
# Add an colunm index "category" with the value "DSTOCK"
n_col = pd.MultiIndex.from_product([DSTOCKV.columns, ["DSTOCKV"]], names=['region', 'category'])
DSTOCKV.columns = n_col

###### OTHER FINAL DEMANDS
# Extract values for each final demand to create variable with dimension region and sector
for rd in exio19.Y.columns.get_level_values("region").unique():
    for ss in exio19.Y.index.get_level_values("sector").unique():
        # caution: it excludes the three last elements of the lst: DINVV (already calculated) DSTOCKV & DVALUEV (sumed with DSTOCKV in df DSTOCKV) & EXPTOTV (always equals zero)
        for var, cmt in zip(lst_upd_FD[:-4], lst_ini_FD[:-4]):
            df_temp = exio19.Y.loc[:,exio19.Y.columns.get_level_values("category") == f"{var}"].groupby(level=1).sum()
            val[f"{BASEYEAR}"] = df_temp.loc[ss, (rd,var)]            
            dict_var[f"{var}_{rd}_{ss}"] = {"val": val.copy(), "cmt": f"{cmt}, nominal", "unit": "current million euros", "idt": "", "eqs": ""}
        val[f"{BASEYEAR}"]  = DSTOCKV.loc[ss, (rd, f"{lst_upd_FD[4]}")] 
        dict_var[f"{lst_upd_FD[4]}_{rd}_{ss}"] = {"val": val.copy() , "cmt": f"{lst_ini_FD[4]}, nominal", "unit": "current million euros", "idt": "", "eqs": ""}
        del df_temp


##################################################
# Total Intermediate consumption - DINTCONV
# Sum to calculate variables
DINTCONSV = exio19.Z.sum()

### CAUTION VERIFY AXIS SUMMATION
# Energy intermediate consumption - DENRTOTV ()
# Caution no bioenergy accounted from crops sector
lst_enr_sect = ["05","06","07","13","14","35","36","37","38","39"]
# Sum demands to energy supply sectors for each region 
DENRTOTV = exio19.Z.loc[(exio19.Z.index.get_level_values('sector').isin(lst_enr_sect))].groupby("sector").sum().sum()

# Calculation of the intermediate consumption demand excluding energy consumption
DMATV = DINTCONSV-DENRTOTV

# Create the variables for each region and each sector                
for r in DINTCONSV.index.get_level_values("region").unique():
    for s in DINTCONSV.index.get_level_values("sector").unique():
        val[f"{BASEYEAR}"] = DINTCONSV.loc[(r,s)]
        dict_var[f"DINTCONSV_{r}_{s}"] = {"val": val.copy(), "cmt": "Total intermediate consumption, nominal", "unit": "current million euros", "idt": "", "eqs": ""}
        val[f"{BASEYEAR}"]  = DENRTOTV.loc[(r,s)]
        dict_var[f"DENRTOTV_{r}_{s}"] = {"val": val.copy(), "cmt": "Total energy consumption, nominal", "unit": "current million euros", "idt": "", "eqs": ""}
        val[f"{BASEYEAR}"]  = DMATV.loc[(r,s)]
        dict_var[f"DMATV_{r}_{s}"] = {"val": val.copy(), "cmt": "Total intermediate consumption except energy, nominal", "unit": "current million euros", "idt": "", "eqs": ""}


#######################################################
# Production - PRODV
PRODV = exio19.x
for r in PRODV.index.get_level_values("region").unique():
    for s in PRODV.index.get_level_values("sector").unique():
            val[f"{BASEYEAR}"]  = PRODV.loc[(r,s), "indout"]
            dict_var[f"PRODV_{r}_{s}"] = {"val": val.copy(), "cmt": "Production, nominal", "unit": "current million euros", "idt": "", "eqs": ""}


# Capital Stock - KSTOCK (using scalar on intenisity previoulsy loaded)
for r in PRODV.index.get_level_values("region").unique():
    for s in PRODV.index.get_level_values("sector").unique():
        val[f"{BASEYEAR}"]  = PRODV.loc[(r,s), "indout"]*int_kstock.loc[(int_kstock["Reg_cd"] == r) & (int_kstock["Sec_cd"] == s), "int_kstock"].sum()
        dict_var[f"KSTOCK_{r}_{s}"] = {"val": val.copy(), "cmt": "Capital stock", "unit": "current million euros", "idt": "", "eqs": ""}


# Value added - VAMPV
VAMPV = exio19.impacts.F.loc[exio19.impacts.F.index.get_level_values("impact") == "Value Added"]
for r in VAMPV.columns.get_level_values("region").unique():
    for s in VAMPV.columns.get_level_values("sector").unique():
            val[f"{BASEYEAR}"] = VAMPV.loc["Value Added", (r,s)]
            dict_var[f"VAMPV_{r}_{s}"] = {"val": val.copy(), "cmt": "Value Added at market price, nominal", "unit": "current million euros", "idt": "", "eqs": ""}


#########################################################
# Variables from Satellite account
# Maps variables from Satelite accounts with NeW variables
lst_vars_stress = ["Taxes less subsidies on products purchased: Total",
"Other net taxes on production",
"Compensation of employees; wages, salaries, & employers' social contributions: Low-skilled",
"Compensation of employees; wages, salaries, & employers' social contributions: Medium-skilled",
"Compensation of employees; wages, salaries, & employers' social contributions: High-skilled",
"Operating surplus: Consumption of fixed capital",
"Operating surplus: Rents on land",
"Operating surplus: Royalties on resources",
"Operating surplus: Remaining net operating surplus",
"Employment: Low-skilled male",
"Employment: Low-skilled female",
"Employment: Medium-skilled male",
"Employment: Medium-skilled female",
"Employment: High-skilled male",
"Employment: High-skilled female",
"Employment hours: Low-skilled male",
"Employment hours: Low-skilled female",
"Employment hours: Medium-skilled male",
"Employment hours: Medium-skilled female",
"Employment hours: High-skilled male",
"Employment hours: High-skilled female",
"Employment: Vulnerable employment",
"Employment hours: Vulnerable employment"]
lst_vars_NeW_stress = ["OTHIMPSUBPRD","TAXPRUDV","CMPEMPALS","CMPEMPAMS","CMPEMPAHS",
"GOPCONSKV","GOPRENTLV","GOPRENTRESV","GOPOTHV",
"DEMPTOTLS_F","DEMPTOTLS_M","DEMPTOTMS_F","DEMPTOTMS_M","DEMPTOTHS_F","DEMPTOTHS_M",
"DEMPHOURLS_F","DEMPHOURLS_M","DEMPHOURMS_F","DEMPHOURMS_M","DEMPHOURHS_F","DEMPHOURHS_M",
"DEMPTOTRISK","DEMPHOURRISK"]

# Add the variable into the dictionnay of data
for r in exio19.satellite.F.columns.get_level_values("region").unique():
    for s in exio19.satellite.F.columns.get_level_values("sector").unique():
            for vars, stress in zip(lst_vars_NeW_stress, lst_vars_stress):
                df_temp = exio19.satellite.F.loc[exio19.satellite.F.index.get_level_values("stressor") == f"{stress}"]
                val[f"{BASEYEAR}"]  = df_temp.loc[f"{stress}", (r,s)]
                if "Employment hours:" in stress: 
                    dict_var[f"{vars}_{r}_{s}"] = {"val": val.copy(), "cmt": stress, "unit": "million hours", "idt": "", "eqs": ""}
                else:
                    if "Employment:" in stress:
                        dict_var[f"{vars}_{r}_{s}"] = {"val": val.copy(), "cmt": stress, "unit": "thousand persons", "idt": "", "eqs": ""}
                    else:
                         dict_var[f"{vars}_{r}_{s}"] = {"val": val.copy(), "cmt": stress, "unit": "million current euros", "idt": "", "eqs": ""}

                    

# Completion of the data with agregates, etc...
CMPEMPA = pd.concat([exio19.satellite.F.loc[exio19.satellite.F.index.get_level_values("stressor") == "Compensation of employees; wages, salaries, & employers' social contributions: Low-skilled"],
                    exio19.satellite.F.loc[exio19.satellite.F.index.get_level_values("stressor") == "Compensation of employees; wages, salaries, & employers' social contributions: Medium-skilled"],
                    exio19.satellite.F.loc[exio19.satellite.F.index.get_level_values("stressor") == "Compensation of employees; wages, salaries, & employers' social contributions: High-skilled"]]).sum()
DEMPTOTLS = pd.concat([exio19.satellite.F.loc[exio19.satellite.F.index.get_level_values("stressor") == "Employment: Low-skilled male"],
                    exio19.satellite.F.loc[exio19.satellite.F.index.get_level_values("stressor") == "Employment: Low-skilled female"]]).sum()
DEMPTOTMS = pd.concat([exio19.satellite.F.loc[exio19.satellite.F.index.get_level_values("stressor") == "Employment: Medium-skilled male"],
                    exio19.satellite.F.loc[exio19.satellite.F.index.get_level_values("stressor") == "Employment: Medium-skilled female"]]).sum()
DEMPTOTHS = pd.concat([exio19.satellite.F.loc[exio19.satellite.F.index.get_level_values("stressor") == "Employment: High-skilled male"],
                    exio19.satellite.F.loc[exio19.satellite.F.index.get_level_values("stressor") == "Employment: High-skilled female"]]).sum()
DEMPTOT = pd.concat([exio19.satellite.F.loc[exio19.satellite.F.index.get_level_values("stressor") == "Employment: Low-skilled male"],
                    exio19.satellite.F.loc[exio19.satellite.F.index.get_level_values("stressor") == "Employment: Low-skilled female"],
                    exio19.satellite.F.loc[exio19.satellite.F.index.get_level_values("stressor") == "Employment: Medium-skilled male"],
                    exio19.satellite.F.loc[exio19.satellite.F.index.get_level_values("stressor") == "Employment: Medium-skilled female"],
                    exio19.satellite.F.loc[exio19.satellite.F.index.get_level_values("stressor") == "Employment: High-skilled male"],
                    exio19.satellite.F.loc[exio19.satellite.F.index.get_level_values("stressor") == "Employment: High-skilled female"]]).sum()
DEMPHOURLS = pd.concat([exio19.satellite.F.loc[exio19.satellite.F.index.get_level_values("stressor") == "Employment hours: Low-skilled male"],
                    exio19.satellite.F.loc[exio19.satellite.F.index.get_level_values("stressor") == "Employment hours: Low-skilled female"]]).sum()
DEMPHOURMS = pd.concat([exio19.satellite.F.loc[exio19.satellite.F.index.get_level_values("stressor") == "Employment hours: Medium-skilled male"],
                    exio19.satellite.F.loc[exio19.satellite.F.index.get_level_values("stressor") == "Employment hours: Medium-skilled female"]]).sum()
DEMPHOURHS = pd.concat([exio19.satellite.F.loc[exio19.satellite.F.index.get_level_values("stressor") == "Employment hours: High-skilled male"],
                    exio19.satellite.F.loc[exio19.satellite.F.index.get_level_values("stressor") == "Employment hours: High-skilled female"]]).sum()
DEMPHOUR = pd.concat([exio19.satellite.F.loc[exio19.satellite.F.index.get_level_values("stressor") == "Employment hours: Low-skilled male"],
                    exio19.satellite.F.loc[exio19.satellite.F.index.get_level_values("stressor") == "Employment hours: Low-skilled female"],
                    exio19.satellite.F.loc[exio19.satellite.F.index.get_level_values("stressor") == "Employment hours: Medium-skilled male"],
                    exio19.satellite.F.loc[exio19.satellite.F.index.get_level_values("stressor") == "Employment hours: Medium-skilled female"],
                    exio19.satellite.F.loc[exio19.satellite.F.index.get_level_values("stressor") == "Employment hours: High-skilled male"],
                    exio19.satellite.F.loc[exio19.satellite.F.index.get_level_values("stressor") == "Employment hours: High-skilled female"]]).sum()

lst_vars_stress_agg =["Compensation of employees: wages, salaries, & employers' social contributions",             
"Employment: Low-skilled","Employment: Medium-skilled","Employment: High-skilled", "Employment",
"Employment hours: Low-skilled","Employment hours: Medium-skilled","Employment hours: High-skilled","Employment hours"]
lst_vars_NeW_stress_agg = ["CMPEMPA","DEMPTOTLS","DEMPTOTMS","DEMPTOTHS","DEMPTOT","DEMPHOURLS","DEMPHOURMS","DEMPHOURHS","DEMPHOUR"]

for r in exio19.satellite.F.columns.get_level_values("region").unique():
    for s in exio19.satellite.F.columns.get_level_values("sector").unique():
            for vars, stress in zip(lst_vars_NeW_stress_agg, lst_vars_stress_agg):
                val[f"{BASEYEAR}"]  = globals()[f"{vars}"].loc[(r,s)]
                if "Employment hours" in stress: 
                    dict_var[f"{vars}_{r}_{s}"] = {"val": val.copy(), "cmt": stress, "unit": "million hours", "idt": "", "eqs": ""}
                else:
                    if "Employment" in stress:
                        dict_var[f"{vars}_{r}_{s}"] = {"val": val.copy(), "cmt": stress, "unit": "thousand persons", "idt": "", "eqs": ""}
                    else:
                         dict_var[f"{vars}_{r}_{s}"] = {"val": val.copy(), "cmt": stress, "unit": "million current euros", "idt": "", "eqs": ""}



19924906.49107241 19924906.49163813


### Trade calculation : Imports & exports

In [11]:
# Bilateral trade flows - IMPV_... from get_gross_trade().bilat_flows

trade_bilat = exio19.get_gross_trade().bilat_flows.copy()
# bilat_flows: df with rows: exporting country and sector, columns: importing countries
trade_bilat = trade_bilat.rename_axis("region_imp", axis=1)

trade_bilat_flat = trade_bilat.stack().reset_index()

trade_bilat_flat.rename(columns={"region": "region_exp", 0: "value"}, inplace=True)

trade_bilat_flat = trade_bilat_flat[["region_imp", "sector", "region_exp", "value"]]

imp_tot = trade_bilat_flat.groupby(["region_imp", "sector"])["value"].sum().copy().reset_index()



# Add bilateral imports in the dictionnary contening variables
for rd in trade_bilat_flat["region_imp"].unique():
    for sd in trade_bilat_flat["sector"].unique():
            for rs in trade_bilat_flat["region_exp"].unique():
                # Bilateral imports by sector
                val[f"{BASEYEAR}"]  = trade_bilat_flat.loc[(trade_bilat_flat["region_imp"] == rd) & (trade_bilat_flat["sector"] == sd) & (trade_bilat_flat["region_exp"] == rs), "value"].values[0]
                dict_var[f"IMPV_{rd}_{sd}_{rs}"]  = {"val": val.copy(), "cmt": f"Imports of sector n°{sd} in {rs} from {rd}, nominal", "unit": "million current euros", "idt": "", "eqs": ""}
            # Total imports by sector
            val[f"{BASEYEAR}"] = imp_tot.loc[(imp_tot["region_imp"] == rd) & (imp_tot["sector"] == sd) , "value"].values[0]
            dict_var[f"IMPV_{rd}_{sd}"]  = {"val": val.copy(), "cmt": f"Imports in {rd} from {rs}, nominal", "unit": "million current euros", "idt": "", "eqs": ""}



c:\Users\bapti\AppData\Local\Programs\Python\Python312\Lib\site-packages\pymrio\tools\iomath.py:610: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  Z_trade_agg = Z_trade_blocks.groupby(axis=1, level=level_spec_Z, sort=False).agg(
c:\Users\bapti\AppData\Local\Programs\Python\Python312\Lib\site-packages\pymrio\tools\iomath.py:610: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  Z_trade_agg = Z_trade_blocks.groupby(axis=1, level=level_spec_Z, sort=False).agg(
c:\Users\bapti\AppData\Local\Programs\Python\Python312\Lib\site-packages\pymrio\tools\iomath.py:613: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  Y_trade_agg = Y_trade_blocks.groupby(axis=1, level=level_spec_Y, sort=False).

### Calculation of the framework of addressed demands
###### (All domestic demands, by type of demand, for industry ss is adressed to domectic sector sd that will thereafter choice to produce or import)

In [12]:
#####################################
### Intermediate consumption
# Total domestic demand
# Demand of region (rd) and sector (sd) to supplying industry (ss) whatever the region of origin (sum of rs)
addci_tot = exio19.Z.groupby("sector", sort=False).sum()
addci_enr_tot = addci_tot.loc[addci_tot.index.get_level_values('sector').isin(lst_enr_sect)]
addci_enr_tot = addci_enr_tot.reindex(addci_tot.index, fill_value=0) 
addci_mat_tot = addci_tot-addci_enr_tot

# Calculate the total intermediate demand addressed to industry "ss" by all industries (sum of sd) from domestic region (rd)
addci_dom = []
addci_enr_dom = []
addci_mat_dom = []
for rd in addci_tot.columns.get_level_values("region").unique():
    for ss in addci_tot.index.get_level_values("sector").unique():
        addci_dom.append(addci_tot.loc[ss,rd].sum())
        addci_enr_dom.append(addci_enr_tot.loc[ss,rd].sum())
        addci_mat_dom.append(addci_mat_tot.loc[ss,rd].sum())
addci_dom = pd.Series(addci_dom, index=addci_tot.columns)
addci_enr_dom = pd.Series(addci_enr_dom, index=addci_enr_tot.columns)
addci_mat_dom = pd.Series(addci_mat_dom, index=addci_mat_tot.columns)


## Calculation of the Intermediate consumption imported 
# Caution: use .copy() to avoid overwriting the initial dataframe
trade_ci = exio19.Z.copy()
# Select only energy demands
trade_ci_enr = exio19.Z.loc[exio19.Z.index.get_level_values('sector').isin(lst_enr_sect)].copy()
trade_ci_enr = trade_ci_enr.reindex(trade_ci.index, fill_value=0) # Reindex the enr dataframe with ci dataframe index and add 0 when none existing values
# Calculation of ci without ener
trade_ci_mat = trade_ci-trade_ci_enr

# Replace values by 0 for domestic intermediate consumption
for region in trade_ci.index.get_level_values('region').unique():
    trade_ci.loc[region, region] = 0
    trade_ci_enr.loc[region, region] = 0
    trade_ci_mat.loc[region, region] = 0

# Imported demand of region (rd) and sector (rd) to supplying industry (ss) whatever the region of origin expect the domestic region (sum of rs != rd)
addciimp_tot = trade_ci.groupby("sector", sort=False).sum()
addciimp_enr_tot = trade_ci_enr.groupby("sector", sort=False).sum()
addciimp_mat_tot = addciimp_tot-addciimp_enr_tot

# Calculate the total imported intermediate demand addressed to industry "ss" by all industries (sum of sd) from domestic region (rd)
addciimp_dom = []
addciimp_enr_dom = []
addciimp_mat_dom = []
for rd in addciimp_tot.columns.get_level_values("region").unique():
    for ss in addciimp_tot.index.get_level_values("sector").unique():
        addciimp_dom.append(addciimp_tot.loc[ss,rd].sum())
        addciimp_enr_dom.append(addciimp_enr_tot.loc[ss,rd].sum())
        addciimp_mat_dom.append(addciimp_mat_tot.loc[ss,rd].sum())
addciimp_dom = pd.Series(addciimp_dom, index=trade_ci.columns)
addciimp_enr_dom = pd.Series(addciimp_enr_dom, index=trade_ci_enr.columns)
addciimp_mat_dom = pd.Series(addciimp_mat_dom, index=trade_ci_mat.columns)

# Add the variables into the model dataset (total intermediate consumption is not included)
for r in addci_dom.index.get_level_values("region").unique():
     for s in addci_dom.index.get_level_values("sector").unique():
        val[f"{BASEYEAR}"]  = addci_enr_dom.loc[r,s]
        dict_var[f"ADDDENRTOTV_{r}_{s}"]  = {"val": val.copy(), "cmt": "Total addressed energy demand, nominal", "unit": "million current euros", "idt": "", "eqs": ""}
        val[f"{BASEYEAR}"]  = addci_mat_dom.loc[r,s]
        dict_var[f"ADDDMATV_{r}_{s}"]  = {"val": val.copy(), "cmt": "Total addressed intermediate consumption (except energy) demand, nominal", "unit": "million current euros", "idt": "", "eqs": ""}
        val[f"{BASEYEAR}"]  = addciimp_enr_dom.loc[r,s]
        dict_var[f"IMPDENRTOTV_{r}_{s}"]  = {"val": val.copy(), "cmt": "Imported energy demand, nominal", "unit": "million current euros", "idt": "", "eqs": ""}
        val[f"{BASEYEAR}"]  = addciimp_mat_dom.loc[r,s]
        dict_var[f"IMPDMATV_{r}_{s}"]  = {"val": val.copy(), "cmt": "Imported intermediate consumption (except energy) demand, nominal", "unit": "million current euros", "idt": "", "eqs": ""}
        

####################################
### Final demands
# Calculation of the Final demands imported
# Caution: use .copy() to avoid overwriting the initial dataframe
# Create a new dataframe summing DSTOCK and DVALUE in DSTOCK and delete EXPTOTV
df_fd = exio19.Y.copy()
for reg in exio19.Y.columns.get_level_values("region").unique():
    df_fd.loc[:,(reg,"DSTOCKV")] =  exio19.Y.loc[:,(reg,"DSTOCKV")]+exio19.Y.loc[:,(reg,"DVALUEV")] 
df_fd = df_fd.loc[:, (df_fd.columns.get_level_values("category") != "DVALUEV") & (df_fd.columns.get_level_values("category") != "EXPTOTV")]

trade_fd = df_fd.copy()
trade_fd_enr = df_fd.loc[df_fd.index.get_level_values('sector').isin(lst_enr_sect)].copy()
trade_fd_enr = trade_fd_enr.reindex(trade_fd.index, fill_value=0)
trade_fd_mat = trade_fd-trade_fd_enr

# Replace values by 0 for domestic final demands
for region in trade_fd.index.get_level_values('region').unique():
    trade_fd.loc[region, region] = 0
    trade_fd_enr.loc[region, region] = 0
    trade_fd_mat.loc[region, region] = 0


# Total domestic demand
# Final demand of region (rd) and sector (sd) to supplying industry (ss) whatever the region of origin (sum of rs)
addfd_temp = df_fd.groupby("sector", sort=False).sum()
# For each final demand, final demand addressed to industry "ss" by domestic region (rd) (permutation of the previous dataframe)
addfd_dom = pd.DataFrame(index=df_fd.index, columns=addfd_temp.columns.get_level_values("category").unique())
for fd in addfd_temp.columns.get_level_values("category").unique():
    for r in addfd_temp.columns.get_level_values("region").unique():
        for s in addfd_temp.index.get_level_values("sector").unique():
            addfd_dom.loc[(r,s),fd] = addfd_temp.loc[s,(r,fd)]


# Imported final demands
# Final demand of region (rd) and sector (sd) to supplying industry (ss) whatever the region of origin expect the domestic region (sum of rs != rd)
addfdimp_tot = trade_fd.groupby("sector", sort=False).sum()
# For each final demand, imported final demand addressed to industry "ss" by domestic region (rd) (permutation of the previous dataframe)
addfdimp_dom = pd.DataFrame(index= trade_fd.index, columns=addfdimp_tot.columns.get_level_values("category").unique())
for fd in addfdimp_tot.columns.get_level_values("category").unique():
    for rd in addfdimp_tot.columns.get_level_values("region").unique():
        for ss in addfdimp_tot.index.get_level_values("sector").unique():
            addfdimp_dom.loc[(rd,ss),fd] = addfdimp_tot.loc[ss,(rd,fd)]

   
# Add the variables into the model dataset
for fd, cmt in zip(addfd_dom.columns.get_level_values("category").unique(),lst_ini_FD[:-2]):
    for r in addfd_dom.index.get_level_values("region").unique():
        for s in addfd_dom.index.get_level_values("sector").unique():
            val[f"{BASEYEAR}"] = addfd_dom.loc[(r,s), fd]
            dict_var[f"ADD{fd}_{r}_{s}"]  = {"val": val.copy(), "cmt":f"Addressed {cmt.lower()}, nominal", "unit": "million current euros", "idt": "", "eqs": ""}
            # Imports by final demand is no more added in the dataset, summed next step
            #val[f"{BASEYEAR}"] = addfdimp_dom.loc[(r,s),fd]
            #dict_var[f"IMPTOTV_{r}_{s}"] = {"val": val.copy(), "cmt": f"Imported {cmt.lower()}, nominal", "unit": "million current euros", "idt": "", "eqs": ""}


# To verify that prod = sum of adressed demands - imports + exports (done for WA_02)
exp_tot = trade_bilat_flat.groupby(["region_exp", "sector"])["value"].sum().copy().reset_index()
EXPTOTV_DE_14 = exp_tot.loc[(exp_tot["region_exp"] == "DE") & (exp_tot["sector"] == "14"), "value"].values[0]
VERIF = (dict_var["PRODV_DE_14"]["val"]["2019Y1"]-(dict_var["ADDDENRTOTV_DE_14"]["val"]["2019Y1"]+dict_var["ADDDMATV_DE_14"]["val"]["2019Y1"]
        +dict_var["ADDCONSHV_DE_14"]["val"]["2019Y1"]+dict_var["ADDCONSNPV_DE_14"]["val"]["2019Y1"]+dict_var["ADDCONSGV_DE_14"]["val"]["2019Y1"]
        +dict_var["ADDDINVV_DE_14"]["val"]["2019Y1"]+dict_var["ADDDSTOCKV_DE_14"]["val"]["2019Y1"]+EXPTOTV_DE_14-dict_var["IMPV_DE_14"]["val"]["2019Y1"]))
print(VERIF)


2.7765054255723953e-08


### Households consumption by purpose


In [13]:
# Parameters to split households energy consumption (to be improved when relevant information available)
param_split = {}

for r in exio19.Y.index.get_level_values("region").unique():
    # Sect 03
    param_split[f"sh_htcl_{r}_03"] = 0.9
    param_split[f"sh_equp_{r}_03"] = (1-param_split[f"sh_htcl_{r}_03"])
    # Sect 14
    param_split[f"sh_htcl_{r}_14"] = 0.1
    param_split[f"sh_icef_{r}_14"] = 0.85
    param_split[f"sh_phef_{r}_14"] = 1-(param_split[f"sh_htcl_{r}_14"]+param_split[f"sh_icef_{r}_14"])
    # Sect 31
    param_split[f"sh_icev_{r}_31"] = 0.6
    param_split[f"sh_phev_{r}_31"] = 0.2
    param_split[f"sh_belv_{r}_31"] = 1-(param_split[f"sh_icev_{r}_31"]+param_split[f"sh_phev_{r}_31"])
    # Sect 32 (idem 31)
    param_split[f"sh_icev_{r}_32"] = param_split[f"sh_icev_{r}_31"]
    param_split[f"sh_phev_{r}_32"] = param_split[f"sh_phev_{r}_31"]
    param_split[f"sh_belv_{r}_32"] = 1-(param_split[f"sh_icev_{r}_32"]+param_split[f"sh_phev_{r}_32"])
    # Sect 35
    param_split[f"sh_htcl_{r}_35"] = 0.9
    param_split[f"sh_phef_{r}_35"]= 0.025
    param_split[f"sh_belf_{r}_35"] = 1-(param_split[f"sh_htcl_{r}_35"]+param_split[f"sh_phef_{r}_35"])
    # Sect 36 (idem 35)
    param_split[f"sh_htcl_{r}_36"] = param_split[f"sh_htcl_{r}_35"]
    param_split[f"sh_phef_{r}_36"] = param_split[f"sh_phef_{r}_35"]
    param_split[f"sh_belf_{r}_36"] = param_split[f"sh_belf_{r}_35"]
    # Sect 37 (idem 35)
    param_split[f"sh_htcl_{r}_37"] = param_split[f"sh_htcl_{r}_35"]
    param_split[f"sh_phef_{r}_37"] = param_split[f"sh_phef_{r}_35"]
    param_split[f"sh_belf_{r}_37"] = param_split[f"sh_belf_{r}_35"]
    # Sect 38
    param_split[f"sh_htcl_{r}_38"] = 0.98
    param_split[f"sh_icef_{r}_38"] = 1-param_split[f"sh_htcl_{r}_38"]


In [14]:
# Specific cases (to split)
for r in exio19.Y.index.get_level_values("region").unique():
    # Heating & Cooling
    val[f"{BASEYEAR}"] = (dict_var[f"CONSHV_{r}_03"]["val"][f"{BASEYEAR}"]*param_split[f"sh_htcl_{r}_03"]
        +dict_var[f"CONSHV_{r}_05"]["val"][f"{BASEYEAR}"]
        +dict_var[f"CONSHV_{r}_06"]["val"][f"{BASEYEAR}"]
        +dict_var[f"CONSHV_{r}_07"]["val"][f"{BASEYEAR}"]
        +dict_var[f"CONSHV_{r}_13"]["val"][f"{BASEYEAR}"]
        +dict_var[f"CONSHV_{r}_14"]["val"][f"{BASEYEAR}"]*param_split[f"sh_htcl_{r}_14"]
        +dict_var[f"CONSHV_{r}_35"]["val"][f"{BASEYEAR}"]*param_split[f"sh_htcl_{r}_35"]
        +dict_var[f"CONSHV_{r}_36"]["val"][f"{BASEYEAR}"]*param_split[f"sh_htcl_{r}_36"]
        +dict_var[f"CONSHV_{r}_37"]["val"][f"{BASEYEAR}"]*param_split[f"sh_htcl_{r}_37"]
        +dict_var[f"CONSHV_{r}_38"]["val"][f"{BASEYEAR}"]*param_split[f"sh_htcl_{r}_38"]
        +dict_var[f"CONSHV_{r}_39"]["val"][f"{BASEYEAR}"])
    dict_var[f"CONSHV_{r}_HTCL"] = {"val": val.copy(), "cmt":f"Households' consumption, Heating & Cooling", "unit": "million current euros", "idt": "", "eqs": ""}
    # ICEVs
    val[f"{BASEYEAR}"] = (dict_var[f"CONSHV_{r}_31"]["val"][f"{BASEYEAR}"]*param_split[f"sh_icev_{r}_31"]
                          +dict_var[f"CONSHV_{r}_32"]["val"][f"{BASEYEAR}"]*param_split[f"sh_icev_{r}_32"])
    dict_var[f"CONSHV_{r}_ICEV"] = {"val": val.copy(), "cmt":f"Households' consumption, Internal consumption engine vehicles", "unit": "million current euros", "idt": "", "eqs": ""}
    # PHEVs
    val[f"{BASEYEAR}"] = (dict_var[f"CONSHV_{r}_31"]["val"][f"{BASEYEAR}"]*param_split[f"sh_phev_{r}_31"]
                          +dict_var[f"CONSHV_{r}_32"]["val"][f"{BASEYEAR}"]*param_split[f"sh_phev_{r}_32"])
    dict_var[f"CONSHV_{r}_PHEV"] = {"val": val.copy(), "cmt":f"Households' consumption, Plug-In Hybrid vehicles", "unit": "million current euros", "idt": "", "eqs": ""}
    # BEVs
    val[f"{BASEYEAR}"] = (dict_var[f"CONSHV_{r}_31"]["val"][f"{BASEYEAR}"]*param_split[f"sh_belv_{r}_31"]
                          +dict_var[f"CONSHV_{r}_32"]["val"][f"{BASEYEAR}"]*param_split[f"sh_belv_{r}_32"])
    dict_var[f"CONSHV_{r}_BELV"] = {"val": val.copy(), "cmt":f"Households' consumption, Battery Electric vehicles", "unit": "million current euros", "idt": "", "eqs": ""}
    # ICE Fuels
    val[f"{BASEYEAR}"] = dict_var[f"CONSHV_{r}_14"]["val"][f"{BASEYEAR}"]*param_split[f"sh_icef_{r}_14"]+dict_var[f"CONSHV_{r}_38"]["val"][f"{BASEYEAR}"]*param_split[f"sh_icef_{r}_38"]
    dict_var[f"CONSHV_{r}_ICEF"] = {"val": val.copy(), "cmt":f"Households' consumption, Internal consumption engine fuels", "unit": "million current euros", "idt": "", "eqs": ""}
    # PHEV Fuels
    val[f"{BASEYEAR}"] = (dict_var[f"CONSHV_{r}_14"]["val"][f"{BASEYEAR}"]*param_split[f"sh_phef_{r}_14"]
                         +dict_var[f"CONSHV_{r}_35"]["val"][f"{BASEYEAR}"]*param_split[f"sh_phef_{r}_35"]
                         +dict_var[f"CONSHV_{r}_36"]["val"][f"{BASEYEAR}"]*param_split[f"sh_phef_{r}_36"]
                         +dict_var[f"CONSHV_{r}_37"]["val"][f"{BASEYEAR}"]*param_split[f"sh_phef_{r}_37"])
    dict_var[f"CONSHV_{r}_PHEF"] = {"val": val.copy(), "cmt":f"Households' consumption, Plug-In Hybrid fuels", "unit": "million current euros", "idt": "", "eqs": ""}
    # BEV Fuels
    val[f"{BASEYEAR}"] = (dict_var[f"CONSHV_{r}_35"]["val"][f"{BASEYEAR}"]*param_split[f"sh_belf_{r}_35"]
                         +dict_var[f"CONSHV_{r}_36"]["val"][f"{BASEYEAR}"]*param_split[f"sh_belf_{r}_36"]
                         +dict_var[f"CONSHV_{r}_37"]["val"][f"{BASEYEAR}"]*param_split[f"sh_belf_{r}_37"])
    dict_var[f"CONSHV_{r}_BELF"] = {"val": val.copy(), "cmt":f"Households' consumption, Battery Electric fuels", "unit": "million current euros", "idt": "", "eqs": ""}

    # EQUP
    lst_equp_oth = ["11","15","16","17","18","19","20","21","22","23","24","25","26","27","28","29","30","33","41"]
    val[f"{BASEYEAR}"] = dict_var[f"CONSHV_{r}_03"]["val"][f"{BASEYEAR}"]*param_split[f"sh_equp_{r}_03"]
    for ss in lst_equp_oth:
        val[f"{BASEYEAR}"] = val[f"{BASEYEAR}"] + dict_var[f"CONSHV_{r}_{ss}"]["val"][f"{BASEYEAR}"]
    dict_var[f"CONSHV_{r}_EQUP"] = {"val": val.copy(), "cmt":f"Households' consumption, Equipments", "unit": "million current euros", "idt": "", "eqs": ""}    

In [15]:
# No need to split
# Mapping dictionnary and aggregates
dict_split_cons = {"FOOD": [["01","02","04","09"],["Food"]], 
                    "CLOT": [["10"],["Clothes"]], 
                    "ESSE": [["FOOD","CLOT"],["Essentials"]], 
                    "RENT": [["52"],["Rent"]], 
                    "WAWA": [["34", "40"],["Wastes & water"]], 
                    "HOUS": [["RENT","WAWA"],["Housing"]],
                    "BASC": [["ESSE","HOUS","HTCL"],["Basics"]],
                    "ICET": [["ICEV","ICEF"],["Internal combustion engine vehicles and fuels"]],
                    "PHET": [["PHEV","PHEF"],["Plug-in hybrid vehicles and fuels"]],
                    "BELT": [["BELV","BELF"],["Battery electric vehicles and fuels"]],
                    "PRVC": [["ICET","PHET","BELT"],["Private vehicles"]], 
                    "RAIL": [["44"],["Railways"]],
                    "OINL": [["45"],["Other inland transport"]],
                    "INLO": [["RAIL","OINL"],["Inland transport expect private cars"]],
                    "AIRT": [["48"],["Air transports"]],
                    "WATT": [["46","47"],["Water transports"]], 
                    "OTTR": [["49"],["Other transports"]],
                    "TRAN": [["PRVC","INLO","AIRT","WATT","OTTR"],["Transports"]],
                    "EDUC": [["57"],["Education"]],
                    "HLTH": [["58"],["Health"]],
                    "ONMS": [["56","59"],["Other non-market services"]],
                    "NMSV": [["EDUC","HLTH","ONMS"],["Non-market services"]],
                    "HRES": [["43"],["Hotels & restaurants"]],
                    "INFC": [["53"],["Information & communication"]],
                    "OLEI": [["12"],["Other leisure"]],
                    "LEIS": [["HRES","INFC","OLEI"],["Leisure"]],
                    "OOTH": [["08","50","51","54","55"],["Others"]],
                    "OTHR": [["LEIS","EQUP","OOTH"],["Other rest"]],
                    "REST": [["OTHR","NMSV"],["Rest"]],
                    "TRAD": [["42"],["Trade"]]}

# Store values into the dictionnary
for r in exio19.Y.index.get_level_values("region").unique():
    for (key, items) in dict_split_cons.items():
        cd = items[0]
        nm = items[1]
        value = 0
        for item in cd:
            value = value + dict_var[f"CONSHV_{r}_{item}"]["val"][f"{BASEYEAR}"]
        val[f"{BASEYEAR}"] = value
        dict_var[f"CONSHV_{r}_{key}"] = {"val": val.copy(), "cmt":f"Households' consumption, {nm[0]}", "unit": "million current euros", "idt": "", "eqs": ""}

### Energy Data - processed in notebook "EnergyBalance_Omnia_v1.00.ipynb"

In [16]:
# Extraction CONSV for Energy (HTCL, ICEF, PHEF, BELF)
rows = []
pattern = r"^CONSHV_.{2}(_HTCL|_ICEF|_PHEF|_BELF)$" # ^ = start, .{2} = 2 strings, (|)= or ,  $ = fin
for k, v in dict_var.items():
    if re.match(pattern, k):
        rows.append((k, v['val']['2019Y1']))

df_var_2019Y1 = pd.DataFrame(rows, columns=["var", "value"])

df_cons_ener = df_var_2019Y1

# ⚠️ line to activate to when the notebook "EnergyBalance_Omnia" is ran stand alone (to get consumption data)
# pathtmp = pathwork + "Data_Raw\EXIOBASE\df_var_2019Y1.csv"
# df_var_2019Y1.to_csv(pathtmp, sep=";", index=False)

In [17]:
# Run notebook doing the energy dataprocessing from Omnia to New in physical and monetary units

path_enr_data = pathwork + "data_raw/energy/"
file_notebook = "EnergyBalance_Omnia_v1.00.ipynb"
full_path = path_enr_data + file_notebook

%run $full_path

df_enr.head(30)

Percentage error between sum of all Omnia and splitted values :  0.0 %


c:\Users\bapti\AppData\Local\Programs\Python\Python312\Lib\site-packages\pymrio\tools\iomath.py:610: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  Z_trade_agg = Z_trade_blocks.groupby(axis=1, level=level_spec_Z, sort=False).agg(
c:\Users\bapti\AppData\Local\Programs\Python\Python312\Lib\site-packages\pymrio\tools\iomath.py:610: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  Z_trade_agg = Z_trade_blocks.groupby(axis=1, level=level_spec_Z, sort=False).agg(
c:\Users\bapti\AppData\Local\Programs\Python\Python312\Lib\site-packages\pymrio\tools\iomath.py:613: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  Y_trade_agg = Y_trade_blocks.groupby(axis=1, level=level_spec_Y, sort=False).

Percentage difference between sum of all Omnia values for DFINENP and splitted values for NeW sectors:  4.841115058702565e-08 %
Percentage difference between sum of all Omnia values for DINTENP and splitted values for NeW sectors:  -1.1974035497851726e-16 %
Percentage difference between sum of all Omnia values for DINTENP and splitted values for NeW sectors:  -1.7219089906972767e-16 %
Verification - Sum of 'Value_splitted' above 'DENRTOTV': 0
Verification - Sum of 'Value_splitted' below 'DENRTOTV': 18
Verification - Sum of 'Value_splitted' above 'DENRTOTV': 0
Verification - Sum of 'Value_splitted' below 'DENRTOTV': 0
Empty DataFrame
Columns: [Value_splitted, DENRTOTV, Verif]
Index: []


,Variable_NeW,Region_NeW,Sector_NeW,Enrp,Year,Unit,Value_melted
0,PENRHT,AU,01,BG,2019,M€/PJ,15.249416
1,PENRHT,AU,02,BG,2019,M€/PJ,15.249416
2,PENRHT,AU,03,BG,2019,M€/PJ,15.249416
3,PENRHT,AU,04,BG,2019,M€/PJ,15.249416
4,PENRHT,AU,15,BG,2019,M€/PJ,15.249416
5,PENRHT,AU,16,BG,2019,M€/PJ,15.249416
6,PENRHT,AU,17,BG,2019,M€/PJ,15.249416
7,PENRHT,AU,41,BG,2019,M€/PJ,15.249416
8,PENRHT,AU,22,BG,2019,M€/PJ,15.249416
9,PENRHT,AU,23,BG,2019,M€/PJ,15.249416


In [18]:
# Suppress doubles
df_enr_physical = df_enr_physical.groupby(by=['Variable_NeW', 'Region_NeW', 'Sector_NeW', 'Enrp', 'Year', 'Unit'], as_index=False)["Value"].sum()

# Suppress doubles for prices and aggregate for quantities
cols = cols = [c for c in df_enr.columns if c not in ['Variable_NeW', 'Value_melted']]
lst_enrv_to_agg = ['DFINENV', 'DINTENV', 'DNONENV']
lst_enrv_to_clean = ['PENRHT', 'TAXENR']

# Sperate dfs
df_enr_to_clean = df_enr[df_enr['Variable_NeW'].isin(lst_enrv_to_clean)]
df_enr_to_agg = df_enr[df_enr['Variable_NeW'].isin(lst_enrv_to_agg)]

# Suppress doubles
df_enr_to_clean = df_enr_to_clean.drop_duplicates(subset=['Variable_NeW'] + cols)

# Aggregate for quantities
df_enr_to_agg = df_enr_to_agg.groupby(['Variable_NeW'] + cols, as_index=False).agg({'Value_melted': 'sum'})

# Merge both
df_enr_clean = pd.concat([df_enr_to_clean, df_enr_to_agg], ignore_index=True)

In [19]:
# df_enr_physical = df_enr_physical.drop(columns=["Year","Unit"]) # Same for all
# Convert dataframe from Energy data processing to NeW variables
# Energy data (physical unit)
dict_varenerp_def = {'DFINENP': "Final energy consumption", 'DINTENP': "Intermediate energy consumption", 
                     'DNONENP': "Non-energy uses", 'IMPP': "Imports", 'EXPP': "Exports", 'DSTOCKP': "Stocks", 'PRODP': "Production"}

for row in df_enr_physical.itertuples(index=False):
    var   = row.Variable_NeW
    reg   = row.Region_NeW
    sec   = row.Sector_NeW
    enrp  = row.Enrp
    unit  = row.Unit
    value = row.Value

    key = f"{var}_{reg}_{sec}_{enrp}"
    dict_var[key] = {
        "val": {BASEYEAR: value},
        "cmt": f"{dict_varenerp_def[var]} of {enrp.lower()} in region {reg} and sector {sec}, physical unit",
        "unit": f"{unit}",
        "idt": "",
        "eqs": ""}


# Energy data (monetary unit)
dict_varener_def = {'DFINENV': "Final energy consumption", 'DINTENV': "Intermediate energy consumption", 
                     'DNONENV': "Non-energy uses", 'PENRHT': "Price without taxes", 'TAXENR': "Taxe"}

for row in df_enr_clean.itertuples(index=False):
    var   = row.Variable_NeW
    reg   = row.Region_NeW
    sec   = row.Sector_NeW
    enrp  = row.Enrp
    unit =  row.Unit
    value = row.Value_melted

    key = f"{var}_{reg}_{sec}_{enrp}"
    dict_var[key] = {
        "val": {BASEYEAR: value},
        "cmt": f"{dict_varener_def[var]} of {enrp.lower()} in region {reg} and sector {sec}, physical unit",
        "unit": f"{unit}",
        "idt": "",
        "eqs": ""}

## Energy Module

#### Split consumption functions by energy source

In [20]:
# Define parameters for split based on previous parameters

for r in exio19.Y.index.get_level_values("region").unique():
    # Sect 03 ["HSBM","EQUP"]
    param_split[f"sh_hsbm_{r}_03"] = param_split[f"sh_htcl_{r}_03"]

    # Sect 07 ["HGAS", "HBGS"]
    param_split[f"sh_hgas_{r}_07"] = (dict_var[f"DFINENV_{r}_HC_GAS"]["val"][f"{BASEYEAR}"])/(dict_var[f"DFINENV_{r}_HC_GAS"]["val"][f"{BASEYEAR}"]+dict_var[f"DFINENV_{r}_HC_BG"]["val"][f"{BASEYEAR}"])
    param_split[f"sh_hbgs_{r}_07"] = (dict_var[f"DFINENV_{r}_HC_BG"]["val"][f"{BASEYEAR}"])/(dict_var[f"DFINENV_{r}_HC_GAS"]["val"][f"{BASEYEAR}"]+dict_var[f"DFINENV_{r}_HC_BG"]["val"][f"{BASEYEAR}"])

    # Sect 14 ["HOIL","IFOI","IFLB","PFOI","PFBF"]
    param_split[f"sh_hoil_{r}_14"] = param_split[f"sh_htcl_{r}_14"]
    param_split[f"sh_ifoi_{r}_14"] = param_split[f"sh_icef_{r}_14"]*(dict_var[f"DFINENV_{r}_TR_OIL"]["val"][f"{BASEYEAR}"])/(dict_var[f"DFINENV_{r}_TR_OIL"]["val"][f"{BASEYEAR}"]+dict_var[f"DFINENV_{r}_TR_LBF"]["val"][f"{BASEYEAR}"])
    param_split[f"sh_iflb_{r}_14"] = param_split[f"sh_icef_{r}_14"]*(dict_var[f"DFINENV_{r}_TR_LBF"]["val"][f"{BASEYEAR}"])/(dict_var[f"DFINENV_{r}_TR_OIL"]["val"][f"{BASEYEAR}"]+dict_var[f"DFINENV_{r}_TR_LBF"]["val"][f"{BASEYEAR}"])
    param_split[f"sh_pfoi_{r}_14"] = param_split[f"sh_phef_{r}_14"]*0.9
    param_split[f"sh_pfbf_{r}_14"] = param_split[f"sh_phef_{r}_14"]*0.1

    # Sect 35 ["HELC","PFEL","BELF"]
    param_split[f"sh_helc_{r}_35"] = param_split[f"sh_htcl_{r}_35"]
    param_split[f"sh_pfel_{r}_35"] = param_split[f"sh_phef_{r}_35"]
    # Sect 36 (idem 35) ["HELC","PFEL","BELF"]
    param_split[f"sh_helc_{r}_36"] = param_split[f"sh_helc_{r}_35"]
    param_split[f"sh_pfel_{r}_36"] = param_split[f"sh_pfel_{r}_35"]
    # Sect 37 (idem 35) ["HELC","PFEL","BELF"]
    param_split[f"sh_helc_{r}_37"] = param_split[f"sh_helc_{r}_35"]
    param_split[f"sh_pfel_{r}_37"] = param_split[f"sh_pfel_{r}_35"]
    # Sect 38 ["HGAS","HBGS","IFGS","IFBG"]
    param_split[f"sh_hgas_{r}_38"] = param_split[f"sh_htcl_{r}_38"]*(dict_var[f"DFINENV_{r}_HC_GAS"]["val"][f"{BASEYEAR}"])/(
                                    dict_var[f"DFINENV_{r}_HC_GAS"]["val"][f"{BASEYEAR}"]+dict_var[f"DFINENV_{r}_HC_BG"]["val"][f"{BASEYEAR}"])
    param_split[f"sh_hbgs_{r}_38"] = param_split[f"sh_htcl_{r}_38"]*(dict_var[f"DFINENV_{r}_HC_BG"]["val"][f"{BASEYEAR}"])/(
                                    dict_var[f"DFINENV_{r}_HC_GAS"]["val"][f"{BASEYEAR}"]+dict_var[f"DFINENV_{r}_HC_BG"]["val"][f"{BASEYEAR}"])
    param_split[f"sh_ifgs_{r}_38"] = param_split[f"sh_icef_{r}_38"]*(dict_var[f"DFINENV_{r}_TR_GAS"]["val"][f"{BASEYEAR}"])/(
                                    dict_var[f"DFINENV_{r}_TR_GAS"]["val"][f"{BASEYEAR}"]+dict_var[f"DFINENV_{r}_TR_BG"]["val"][f"{BASEYEAR}"])
    param_split[f"sh_ifbg_{r}_38"] = param_split[f"sh_icef_{r}_38"]*(dict_var[f"DFINENV_{r}_TR_BG"]["val"][f"{BASEYEAR}"])/(
                                    dict_var[f"DFINENV_{r}_TR_GAS"]["val"][f"{BASEYEAR}"]+dict_var[f"DFINENV_{r}_TR_BG"]["val"][f"{BASEYEAR}"])
    
    # Sect 39 ["HHEA", "HGEO", "HSLT"]
    heat_total = (dict_var[f"DFINENV_{r}_HC_HEAT"]["val"][f"{BASEYEAR}"]+dict_var[f"DFINENV_{r}_HC_GEO"]["val"][f"{BASEYEAR}"]
                             +dict_var[f"DFINENV_{r}_HC_SOLT"]["val"][f"{BASEYEAR}"])
    if heat_total > 0.01:
        param_split[f"sh_hhea_{r}_39"] = (dict_var[f"DFINENV_{r}_HC_HEAT"]["val"][f"{BASEYEAR}"])/(dict_var[f"DFINENV_{r}_HC_HEAT"]["val"][f"{BASEYEAR}"]
                                            +dict_var[f"DFINENV_{r}_HC_GEO"]["val"][f"{BASEYEAR}"]+dict_var[f"DFINENV_{r}_HC_SOLT"]["val"][f"{BASEYEAR}"])
        param_split[f"sh_hgeo_{r}_39"] = (dict_var[f"DFINENV_{r}_HC_GEO"]["val"][f"{BASEYEAR}"])/(dict_var[f"DFINENV_{r}_HC_HEAT"]["val"][f"{BASEYEAR}"]
                                            +dict_var[f"DFINENV_{r}_HC_GEO"]["val"][f"{BASEYEAR}"]+dict_var[f"DFINENV_{r}_HC_SOLT"]["val"][f"{BASEYEAR}"])
        param_split[f"sh_hslt_{r}_39"] = (dict_var[f"DFINENV_{r}_HC_SOLT"]["val"][f"{BASEYEAR}"])/(dict_var[f"DFINENV_{r}_HC_HEAT"]["val"][f"{BASEYEAR}"]
                                        +dict_var[f"DFINENV_{r}_HC_GEO"]["val"][f"{BASEYEAR}"]+dict_var[f"DFINENV_{r}_HC_SOLT"]["val"][f"{BASEYEAR}"])
    else:
        param_split[f"sh_hhea_{r}_39"] = 0.8
        param_split[f"sh_hgeo_{r}_39"] = 0.1
        param_split[f"sh_hslt_{r}_39"] = 0.1




In [21]:
# Calculation of consumption purposes for energy

for r in exio19.Y.index.get_level_values("region").unique():
    ######
    # ICEF: Internal Combustion Engine Fuels
    ###
    ## ILQF: Liquid fuels
    val[f"{BASEYEAR}"] = dict_var[f"CONSHV_{r}_14"]["val"][f"{BASEYEAR}"]*param_split[f"sh_icef_{r}_14"]
    dict_var[f"CONSHV_{r}_ILQF"] = {"val": val.copy(), "cmt":f"Households' consumption, ICE, liquid fuels for private transport", "unit": "million current euros", "idt": "", "eqs": ""}
    # IFOI: Oil fuels
    val[f"{BASEYEAR}"] =  dict_var[f"CONSHV_{r}_14"]["val"][f"{BASEYEAR}"]*param_split[f"sh_ifoi_{r}_14"]
    dict_var[f"CONSHV_{r}_IFOI"] = {"val": val.copy(), "cmt":f"Households' consumption, ICE, oil fuels", "unit": "million current euros", "idt": "", "eqs": ""}
    # IFBF: Liquid bio-fuels
    val[f"{BASEYEAR}"] =  dict_var[f"CONSHV_{r}_14"]["val"][f"{BASEYEAR}"]*param_split[f"sh_iflb_{r}_14"]
    dict_var[f"CONSHV_{r}_IFLB"] = {"val": val.copy(), "cmt":f"Households' consumption, ICE, liquid bio-fuels", "unit": "million current euros", "idt": "", "eqs": ""}

    ###
    ## IGSF : Gaseous fuels
    val[f"{BASEYEAR}"] = dict_var[f"CONSHV_{r}_38"]["val"][f"{BASEYEAR}"]*(param_split[f"sh_ifgs_{r}_38"]+param_split[f"sh_ifbg_{r}_38"])
    dict_var[f"CONSHV_{r}_IGSF"] = {"val": val.copy(), "cmt":f"Households' consumption, ICE, gaseous fuels", "unit": "million current euros", "idt": "", "eqs": ""}
    # IFGS: Fossil gaseous fuels
    val[f"{BASEYEAR}"] =  dict_var[f"CONSHV_{r}_38"]["val"][f"{BASEYEAR}"]*param_split[f"sh_ifgs_{r}_38"]
    dict_var[f"CONSHV_{r}_IFGS"] = {"val": val.copy(), "cmt":f"Households' consumption, ICE, fossil gaseous", "unit": "million current euros", "idt": "", "eqs": ""}
    # IFBF: Liquid bio-fuels
    val[f"{BASEYEAR}"] =  dict_var[f"CONSHV_{r}_38"]["val"][f"{BASEYEAR}"]*param_split[f"sh_ifbg_{r}_38"]
    dict_var[f"CONSHV_{r}_IFBG"] = {"val": val.copy(), "cmt":f"Households' consumption, ICE, biogas", "unit": "million current euros", "idt": "", "eqs": ""}


    ######
    # PHEF: Plug-in Hybrid-Electric Fuels
    ###
    # PFOI: Oil fuels
    val[f"{BASEYEAR}"] = dict_var[f"CONSHV_{r}_14"]["val"][f"{BASEYEAR}"]*param_split[f"sh_pfoi_{r}_14"]
    dict_var[f"CONSHV_{r}_PFOI"] = {"val": val.copy(), "cmt":f"Households' consumption, PHEV, oil", "unit": "million current euros", "idt": "", "eqs": ""}
    ###
    # PFBF: Liquid biofuels
    val[f"{BASEYEAR}"] = dict_var[f"CONSHV_{r}_14"]["val"][f"{BASEYEAR}"]*param_split[f"sh_pfbf_{r}_14"]
    dict_var[f"CONSHV_{r}_PFBF"] = {"val": val.copy(), "cmt":f"Households' consumption, PHEV, bio-fuels", "unit": "million current euros", "idt": "", "eqs": ""}
    ###
    # PFEL: Elec fuels
    val[f"{BASEYEAR}"] = (dict_var[f"CONSHV_{r}_35"]["val"][f"{BASEYEAR}"]*param_split[f"sh_pfel_{r}_35"]
                         +dict_var[f"CONSHV_{r}_36"]["val"][f"{BASEYEAR}"]*param_split[f"sh_pfel_{r}_36"]
                         +dict_var[f"CONSHV_{r}_37"]["val"][f"{BASEYEAR}"]*param_split[f"sh_pfel_{r}_37"])
    dict_var[f"CONSHV_{r}_PFEL"] = {"val": val.copy(), "cmt":f"Households' consumption, PHEV, electricity", "unit": "million current euros", "idt": "", "eqs": ""}
    

    ######
    # HTCL: Heating and cooling
    # HSBM: Solid Biomass
    val[f"{BASEYEAR}"] = dict_var[f"CONSHV_{r}_03"]["val"][f"{BASEYEAR}"]*param_split[f"sh_hsbm_{r}_03"]
    dict_var[f"CONSHV_{r}_HSBM"] = {"val": val.copy(), "cmt":f"Households' consumption, Heating & Cooling, Solid Biomass", "unit": "million current euros", "idt": "", "eqs": ""}
    # HCOM: Solid fossil fuels
    val[f"{BASEYEAR}"] = dict_var[f"CONSHV_{r}_05"]["val"][f"{BASEYEAR}"]+dict_var[f"CONSHV_{r}_13"]["val"][f"{BASEYEAR}"]
    dict_var[f"CONSHV_{r}_HCOM"] = {"val": val.copy(), "cmt":f"Households' consumption, Heating & Cooling, Solid Fossil fuels", "unit": "million current euros", "idt": "", "eqs": ""}
    # HOIL: Oil
    val[f"{BASEYEAR}"] = dict_var[f"CONSHV_{r}_06"]["val"][f"{BASEYEAR}"]+dict_var[f"CONSHV_{r}_14"]["val"][f"{BASEYEAR}"]*param_split[f"sh_hoil_{r}_14"]
    dict_var[f"CONSHV_{r}_HOIL"] = {"val": val.copy(), "cmt":f"Households' consumption, Heating & Cooling, Solid Fossil fuels", "unit": "million current euros", "idt": "", "eqs": ""}
    # HGAS: Fossil gaseous fuels
    val[f"{BASEYEAR}"] = dict_var[f"CONSHV_{r}_07"]["val"][f"{BASEYEAR}"]*param_split[f"sh_hgas_{r}_07"]+dict_var[f"CONSHV_{r}_38"]["val"][f"{BASEYEAR}"]*param_split[f"sh_hgas_{r}_38"]
    dict_var[f"CONSHV_{r}_HGAS"] = {"val": val.copy(), "cmt":f"Households' consumption, Heating & Cooling, Fossil gaseous fuels", "unit": "million current euros", "idt": "", "eqs": ""}
    # HBGS: Biogas
    val[f"{BASEYEAR}"] = dict_var[f"CONSHV_{r}_07"]["val"][f"{BASEYEAR}"]*param_split[f"sh_hbgs_{r}_07"]+dict_var[f"CONSHV_{r}_38"]["val"][f"{BASEYEAR}"]*param_split[f"sh_hbgs_{r}_38"]
    dict_var[f"CONSHV_{r}_HBGS"] = {"val": val.copy(), "cmt":f"Households' consumption, Heating & Cooling, biofuels", "unit": "million current euros", "idt": "", "eqs": ""}
    # HELC: Electricity
    val[f"{BASEYEAR}"] = (dict_var[f"CONSHV_{r}_35"]["val"][f"{BASEYEAR}"]*param_split[f"sh_helc_{r}_35"]
                        +dict_var[f"CONSHV_{r}_36"]["val"][f"{BASEYEAR}"]*param_split[f"sh_helc_{r}_36"]
                        +dict_var[f"CONSHV_{r}_37"]["val"][f"{BASEYEAR}"]*param_split[f"sh_helc_{r}_37"])
    dict_var[f"CONSHV_{r}_HELC"] = {"val": val.copy(), "cmt":f"Households' consumption, Heating & Cooling, Electricity", "unit": "million current euros", "idt": "", "eqs": ""}
    
    # Verify total heat not null
    heat_total = (dict_var[f"DFINENV_{r}_HC_HEAT"]["val"][f"{BASEYEAR}"]+dict_var[f"DFINENV_{r}_HC_GEO"]["val"][f"{BASEYEAR}"]
                            +dict_var[f"DFINENV_{r}_HC_SOLT"]["val"][f"{BASEYEAR}"])
    # HHEA: Heat
    val[f"{BASEYEAR}"] = dict_var[f"CONSHV_{r}_39"]["val"][f"{BASEYEAR}"]*param_split[f"sh_hhea_{r}_39"]
    dict_var[f"CONSHV_{r}_HHEA"] = {"val": val.copy(), "cmt":f"Households' consumption, Heating & Cooling, Heat", "unit": "million current euros", "idt": "", "eqs": ""}
    # HGEO: Geothermal
    val[f"{BASEYEAR}"] = dict_var[f"CONSHV_{r}_39"]["val"][f"{BASEYEAR}"]*param_split[f"sh_hgeo_{r}_39"]
    dict_var[f"CONSHV_{r}_HGEO"] = {"val": val.copy(), "cmt":f"Households' consumption, Heating & Cooling, Geothermal", "unit": "million current euros", "idt": "", "eqs": ""}
    # HSLT: Solar Thermal
    val[f"{BASEYEAR}"] = dict_var[f"CONSHV_{r}_39"]["val"][f"{BASEYEAR}"]*param_split[f"sh_hslt_{r}_39"]
    dict_var[f"CONSHV_{r}_HSLT"] = {"val": val.copy(), "cmt":f"Households' consumption, Heating & Cooling, Solar Thermal", "unit": "million current euros", "idt": "", "eqs": ""}
         


### GHG Data & Scalars

In [22]:
pathghg = pathwork + "data_raw/ghg_emissions/"
fl_nm_var = "data_ghg.var"
fl_nm_scl = "scl_ghg.scl"
save_ghg_var = pathghg+fl_nm_var
save_ghg_scl = pathghg+fl_nm_scl

io.variables.load(save_ghg_var)
io.scalars.load(save_ghg_scl)

SMP = SMPSTRY + ":" + SMPENDY
io.variables.sample = f'{SMP}'

Loading E:/Work/DIAMOND/NeW/data_raw/ghg_emissions/data_ghg.var
5566 objects loaded
Loading E:/Work/DIAMOND/NeW/data_raw/ghg_emissions/scl_ghg.scl
18 objects loaded


#### Store variables and comments into iode objects

In [23]:
for key in dict_var.keys():
    io.variables[f"{key}"] = np.nan
    io.variables[f"{key}", f"{LOADYEAR}"] = dict_var[f"{key}"]["val"][f"{LOADYEAR}"]
    io.comments[f"{key}"] = dict_var[f"{key}"]["cmt"] + "; Unit:" + dict_var[f"{key}"]["unit"]  
io.variables    

Workspace: Variables
nb variables: 154002
filename: e:\Work\DIAMOND\NeW\data_raw\ghg_emissions\data_ghg.var
sample: 2015Y1:2050Y1
mode: LEVEL

      name     	2015Y1	2016Y1	2017Y1	2018Y1	  2019Y1 	...	2045Y1	2046Y1	2047Y1	2048Y1	2049Y1	2050Y1
ADDCONSGV_AU_01	    na	    na	    na	    na	   252.29	...	    na	    na	    na	    na	    na	    na
ADDCONSGV_AU_02	    na	    na	    na	    na	   207.06	...	    na	    na	    na	    na	    na	    na
ADDCONSGV_AU_03	    na	    na	    na	    na	    80.16	...	    na	    na	    na	    na	    na	    na
ADDCONSGV_AU_04	    na	    na	    na	    na	    22.42	...	    na	    na	    na	    na	    na	    na
ADDCONSGV_AU_05	    na	    na	    na	    na	    66.29	...	    na	    na	    na	    na	    na	    na
...            	   ...	   ...	   ...	   ...	      ...	...	   ...	   ...	   ...	   ...	   ...	   ...
VAMPV_WM_55    	    na	    na	    na	    na	163573.78	...	    na	    na	    na	    na	    na	    na
VAMPV_WM_56    	    na	    na	    na	    na	222696.27	...

#### Completing data 

In [24]:
# Complete missing variables by zero
# List of NeW sectors for each emissions sector (same for each gases, except F-Gases) 
lst_sec_dfinen = [f"{i:02}" for i in range(1, 60)] + ["HC", "TR"]
lst_enrp_dfinen = ["BG","COMB","ELEC","GAS","GEO","HEAT","IW","LBF","OIL","SBM","SOLT"]
dict_lst_sec_dinten = {"13": ["COMB","OIL","GAS"], 
                       "14": ["COMB","GAS","OIL"], 
                       "17": ["OIL", "SBM", "GAS", "COMB"],
                      "22": ["COMB","OIL","GAS"],
                        "35": ["BG","COMB","GAS","GEO","IW","LBF","NUC","OIL","SBM","SOLT"],
                        "36": ["BG","COMB","GAS","GEO","IW","LBF","NUC","OIL","SBM","SOLT"], 
                        "37": ["BG","COMB","GAS","GEO","IW","LBF","NUC","OIL","SBM","SOLT"],
                        '38': ["COMB","GAS","OIL"],
                        "39": ["BG","COMB","GAS","GEO","IW","LBF","NUC","OIL","SBM","SOLT"]}

dict_lst_sec_dnonen =  {"15": ["COMB","OIL","GAS"], "17": ["COMB","GAS","OIL"], "41": ["OIL", "GAS", "COMB"]}

lst_pg_src = ["COMB","CCCS","GAS","GCCS","OIL","NUC","IW","BG","SBM","BCCS","HYD","GEO","PV","SOLT","WDON", "WDOF","TID"]


# DFINENP & DFINENV
dfp = io.variables[f"DFINENP_*_*_*"].to_frame().reset_index()
dfv = io.variables[f"DFINENV_*_*_*"].to_frame().reset_index()
for reg in lst_reg:
    for sec in lst_sec_dfinen:
        for enrp in lst_enrp_dfinen:
            dfp_tmp = dfp.loc[dfp["names"] == f"DFINENP_{reg}_{sec}_{enrp}"]
            if dfp_tmp.empty:
                io.variables[f"DFINENP_{reg}_{sec}_{enrp}"] = 0
            dfv_tmp = dfv.loc[dfv["names"] == f"DFINENV_{reg}_{sec}_{enrp}"]
            if dfv_tmp.empty:
                io.variables[f"DFINENV_{reg}_{sec}_{enrp}"] = 0

# DINTENP & DINTENV
dfp = io.variables[f"DINTENP_*_*_*"].to_frame().reset_index()
dfv = io.variables[f"DINTENV_*_*_*"].to_frame().reset_index()
for reg in lst_reg:
    for sec, lst_tmp in dict_lst_sec_dinten.items():
        for enrp in lst_tmp:
            dfp_tmp = dfp.loc[dfp["names"] == f"DINTENP_{reg}_{sec}_{enrp}"]
            if dfp_tmp.empty:
                io.variables[f"DINTENP_{reg}_{sec}_{enrp}"] = 0
            dfv_tmp = dfv.loc[dfv["names"] == f"DINTENV_{reg}_{sec}_{enrp}"]
            if dfv_tmp.empty:
                io.variables[f"DINTENV_{reg}_{sec}_{enrp}"] = 0
                

# DNONENP & DNONENV
dfp = io.variables[f"DNONENP_*_*_*"].to_frame().reset_index()
dfv = io.variables[f"DNONENV_*_*_*"].to_frame().reset_index()
for reg in lst_reg:
    for sec, lst_tmp in dict_lst_sec_dnonen.items():
        for enrp in lst_tmp:
            dfp_tmp = dfp.loc[dfp["names"] == f"DNONENP_{reg}_{sec}_{enrp}"]
            if dfp_tmp.empty:
                io.variables[f"DNONENP_{reg}_{sec}_{enrp}"] = 0
            dfv_tmp = dfv.loc[dfv["names"] == f"DNONENV_{reg}_{sec}_{enrp}"]
            if dfv_tmp.empty:
                io.variables[f"DNONENV_{reg}_{sec}_{enrp}"] = 0     


# PENRHT / TAXENR 
dfp = io.variables[f"PENRHT_*_*_*"].to_frame().reset_index()
dft = io.variables[f"TAXENR_*_*_*"].to_frame().reset_index()
for reg in lst_reg:
    for sec in lst_sec_dfinen:
        for enrp in lst_enrp_dfinen:
            dfp_tmp = dfp.loc[dfp["names"] == f"PENRHT_{reg}_{sec}_{enrp}"]
            if dfp_tmp.empty:
                io.variables[f"PENRHT_{reg}_{sec}_{enrp}"] = 15       
            dft_tmp = dft.loc[dft["names"] == f"TAXENR_{reg}_{sec}_{enrp}"]
            if dft_tmp.empty:
                io.variables[f"TAXENR_{reg}_{sec}_{enrp}"] = 0                         


In [25]:
# Aggregate DFINENV, DINTENV, DNONENV for all energy products (Enrp)
# DFINENV
for r in lst_reg:
    for sec in lst_sec_dfinen:
        exp = []
        for enrp in lst_enrp_dfinen:
            exp.append(f"DFINENV_{r}_{sec}_{enrp}")
        io.identities[f"DFINENV_{r}_{sec}"] = "+".join(exp)
        io.identities.execute(f"DFINENV_{r}_{sec}", f"{BASEYEAR}", f"{BASEYEAR}")

# DINTENV
for r in lst_reg:
    for sec, lst_temp in dict_lst_sec_dinten.items():
        exp = [] 
        for enrp in lst_temp:
            exp.append(f"DINTENV_{r}_{sec}_{enrp}")
        io.identities[f"DINTENV_{r}_{sec}"] = "+".join(exp)
        io.identities.execute(f"DINTENV_{r}_{sec}", f"{BASEYEAR}", f"{BASEYEAR}")  

# DNONENR
for r in lst_reg:
    for sec, lst_temp in dict_lst_sec_dnonen.items():
        exp = [] 
        for enrp in lst_temp:
            exp.append(f"DNONENV_{r}_{sec}_{enrp}")
        io.identities[f"DNONENV_{r}_{sec}"] = "+".join(exp)
        io.identities.execute(f"DNONENV_{r}_{sec}", f"{BASEYEAR}", f"{BASEYEAR}")          
    


#### Sum variables by region and run identities

In [26]:
# List of variable created
sub_set = io.variables["*_FR_35*"]
sub_set.names
lst_var_toagg = []
for var in sub_set.names:
    lst_var_toagg.append(f"{var[:-6]}")

to_remove = ("DFINENP", "DFINENV", "DINTENP", "DINTENV", "PRODP", "PENRHT", "TAXENR", "EM")

lst_var_toagg = [x for x in lst_var_toagg if not x.startswith(to_remove)]

# Suppress IMPV_FR form the list from IMPV_??_??_?? variables, only EXPTOTV remains
pattern = re.compile(r"^IMPV_FR")
lst_var_toagg = [item for item in lst_var_toagg if not pattern.match(item)] 

print(lst_var_toagg)  

for var in lst_var_toagg:
    for r in exio19.satellite.F.columns.get_level_values("region").unique():
        tmp = "+".join(f"{var}_{r}_{s}" for s in exio19.satellite.F.columns.get_level_values("sector").unique())
        io.identities[f"{var}_{r}"] = tmp
        io.identities.execute(f"{var}_{r}", "2019Y1", "2019Y1")

['ADDCONSGV', 'ADDCONSHV', 'ADDCONSNPV', 'ADDDENRTOTV', 'ADDDINVV', 'ADDDMATV', 'ADDDSTOCKV', 'CMPEMPAHS', 'CMPEMPALS', 'CMPEMPAMS', 'CMPEMPA', 'CONSGV', 'CONSHV', 'CONSNPV', 'DEMPHOURHS', 'DEMPHOURHS_F', 'DEMPHOURHS_M', 'DEMPHOURLS', 'DEMPHOURLS_F', 'DEMPHOURLS_M', 'DEMPHOURMS', 'DEMPHOURMS_F', 'DEMPHOURMS_M', 'DEMPHOURRISK', 'DEMPHOUR', 'DEMPTOTHS', 'DEMPTOTHS_F', 'DEMPTOTHS_M', 'DEMPTOTLS', 'DEMPTOTLS_F', 'DEMPTOTLS_M', 'DEMPTOTMS', 'DEMPTOTMS_F', 'DEMPTOTMS_M', 'DEMPTOTRISK', 'DEMPTOT', 'DENRTOTV', 'DINTCONSV', 'DINVV', 'DMATV', 'DSTOCKV', 'GOPCONSKV', 'GOPOTHV', 'GOPRENTLV', 'GOPRENTRESV', 'IMPDENRTOTV', 'IMPDMATV', 'IMPV', 'KSTOCK', 'OTHIMPSUBPRD', 'PRODV', 'TAXPRUDV', 'VAMPV']


### Calculation of HHs accounts

In [27]:
dict_var_sa = dict()

# Loading pre-calculated scalars (from OECD "" and processed)
path_scl_hhinc = pathwork + "data_raw/non-fin_accounts/"
csv_scl_hhinc = os.path.join(path_scl_hhinc, "HH_scl_inc_2019.csv")
scl_hhinc = pd.read_csv(csv_scl_hhinc, sep=";", dtype={0: str}, header=[0], index_col=[0])

# Calculation of variables proportional to CMPEMPA
# Sum all CMPEMPA by country
CMPEMPA_r = CMPEMPA.groupby("region").sum()
# Variables and comments
dict_var_cmph = {"CMPEMREH": "Compensation of employess (D1), received by households", "CMPEMPAH": "Compensation of employess (D1), paid by households", 
                "INCPTYREH": "Property income (D4), received by Households", "INCPTYPAH": "Property income (D4), paid by Households",
                "TXDIRPAH": "Current taxes on income and wealth (D5), paid by Households", 
                "SCNETREH": "Net social contribution (D61), received by Households", "SCNETPAH": "Net social contribution (D61), paid by Households",
                "SOCBENOTHREH": "Social benefits other than transfers in kind (D62), received by Households", "SOCBENOTHPAH": "Social benefits other than transfers in kind (D62), paid by Households",
                "SOCBENKIREH": "Social benefits in kind (D63), received by households", "SOCBENKIPAH": "Social benefits in kind (D63), paid by households",
                "CURTRANSREH": "Other current transfers (D7), received by Households", "CURTRANSPAH": "Other current transfers (D7), received by Households",
                "INCGDISPH": "Households' gross disposable income (B6G)"}

for reg in CMPEMPA_r.index.unique():
    for key, value in dict_var_cmph.items():
        scl_nm = f"sh{key.lower()}"
        var_nm = f"{key}_{reg}"
        val[f"{BASEYEAR}"] = CMPEMPA_r.loc[(reg)].sum()*scl_hhinc.loc[(reg,scl_nm)].sum()
        dict_var_sa[var_nm]  = {"val": val.copy(), "cmt": value, "unit": "million current euros", "idt": "", "eqs": ""}


# Calculation of variables proportional to INCGDISP
dict_var_inch = {"ADJPENREH": "Adjustment for the change in pension entitlements (D8), received by Households", "ADJPENPAH": "Adjustment for the change in pension entitlements (D8), paid by Households",
                "GSAVH": "Households' gross saving (B8G)",
                "KTRANSREH": "Capital transfers (D9), received by Households", "KTRANSPAH": "Capital transfers (D9), paid by Households",
                "FIXCAPPAH": "Households' gross capital formation (P5)",
                "DSTOCKVH": "Changes in inventories (P52), for Households",
                "DVALUEVH": "Acquisitions less disposals of valuables (P53) for Households", 
                "ACLDNFNPAH": "Acquisitions less disposals of non-financial non-produced assets (NP), paid by Households",
                "NLENDBH": "Households' net lending (+) / net borrowing (-)"}

for reg in CMPEMPA_r.index.unique():
    for key, value in dict_var_inch.items():
        scl_nm = f"sh{key.lower()}"
        var_nm = f"{key}_{reg}"
        var_inc = f"INCGDISPH_{reg}"
        val[f"{BASEYEAR}"] = dict_var_sa[var_inc]["val"][f"{BASEYEAR}"]*scl_hhinc.loc[(reg,scl_nm)].sum()
        dict_var_sa[var_nm]  = {"val": val.copy(), "cmt": value, "unit": "million current euros", "idt": "", "eqs": ""}

# Remaining variables
op_var = ["Operating surplus: Consumption of fixed capital", "Operating surplus: Rents on land", "Operating surplus: Royalties on resources", "Operating surplus: Remaining net operating surplus"]
GOPSUP = exio19.satellite.F.loc[exio19.satellite.F.index.get_level_values("stressor").isin(op_var)]
GOPSUP_r = GOPSUP.sum().groupby(["region"]).sum()
for reg in GOPSUP_r.index.unique():
    var_nm=  f"GOPSUPH_{reg}"
    val[f"{BASEYEAR}"] = scl_hhinc.loc[(reg,"shgopsuph")].sum()*GOPSUP_r.loc[(reg)].sum()
    dict_var_sa[var_nm]  = {"val": val.copy(), "cmt": "Households' gross operating surplus and mixed income (B2G & B3G)", "unit": "million current euros", "idt": "", "eqs": ""}

DINVV_r = mat_gfcf_dinv.groupby("col_reg").sum()
for reg in DINVV_r.index.unique():
    var_nm=  f"DINVVH_{reg}"
    val[f"{BASEYEAR}"] = scl_hhinc.loc[(reg,"shdinvvh")].sum()*DINVV_r.loc[(reg)].sum()
    dict_var_sa[var_nm]  = {"val": val.copy(), "cmt": "Households' gross fixed capital formation", "unit": "million current euros", "idt": "", "eqs": ""}


## CAUTION
# Re-calculation of DSTOCKVH & NLENDBH to ensure consistency 
for reg in DINVV_r.index.unique():
    val[f"{BASEYEAR}"] = dict_var_sa[f"FIXCAPPAH_{reg}"]["val"][f"{BASEYEAR}"]-dict_var_sa[f"DINVVH_{reg}"]["val"][f"{BASEYEAR}"] 
    var_nm = f"DSTOCKVH_{reg}"
    dict_var_sa[var_nm]  = {"val": val.copy(), "cmt": "Households' changes in inventories (P52) and acquisitions less disposals of valuables (P53)", "unit": "million current euros", "idt": "", "eqs": ""}
    val[f"{BASEYEAR}"] = dict_var_sa[f"GSAVH_{reg}"]["val"][f"{BASEYEAR}"]+dict_var_sa[f"KTRANSREH_{reg}"]["val"][f"{BASEYEAR}"]-dict_var_sa[f"KTRANSPAH_{reg}"]["val"][f"{BASEYEAR}"]-dict_var_sa[f"DINVVH_{reg}"]["val"][f"{BASEYEAR}"]-dict_var_sa[f"DSTOCKVH_{reg}"]["val"][f"{BASEYEAR}"]-dict_var_sa[f"ACLDNFNPAH_{reg}"]["val"][f"{BASEYEAR}"]
    var_nm = f"NLENDBH_{reg}"
    dict_var_sa[var_nm]  = {"val": val.copy(), "cmt": "Households' net lending (+) / net borrowing (-))", "unit": "million current euros", "idt": "", "eqs": ""}


#### Calculation of Goverment accounts

In [28]:
# Loading pre-calculated scalars (from OECD "" and processed)
path_scl_hhinc = pathwork + "data_raw/non-fin_accounts/"
csv_scl_govinc = os.path.join(path_scl_hhinc, "GOV_scl_inc_2019.csv")
scl_govinc = pd.read_csv(csv_scl_govinc, sep=";", dtype={0: str}, header=[0], index_col=[0])

# Calculation of variables proportional to CONSVG
# Sum all CMPEMPA by country
CONSGV_r = exio19.Y.loc[:,(exio19.Y.columns.get_level_values("category") == "CONSGV")].sum()
for reg in CONSGV_r.index.get_level_values("region").unique():
    var_nm = f"CONSGV_{reg}"
    val[f"{BASEYEAR}"] = CONSGV_r.loc[(reg)].sum()
    dict_var_sa[var_nm]  = {"val": val.copy(), "cmt": "Total government' final expenditure", "unit": "million current euros", "idt": "", "eqs": ""}

# Variables and comments
dict_var_consg = {"TXPROIMPREG": "Taxes on production and imports (D2), received by Government", 
                 "SUBTOTPAG": "Subsidies on production and imports (D3), paid by Government",
                 "INCPTYREG": "Property income (D4), received by Government",
                 "INCPTYPAG": "Property income (D4), paid by Government",
                 "TXDIRREG": "Current taxes on income and wealth (D5), received by Government",
                 "TXDIRPAG": "Current taxes on income and wealth (D5), paid by Government",
                 "SCNETREG": "Net social contribution (D61), received by Government",
                 "SCNETPAG": "Net social contribution (D61), paid by Government",
                 "SOCBENOTHREG": "Social benefits other than transfers in kind (D62), received by Government",
                 "SOCBENOTHPAG": "Social benefits other than transfers in kind (D62), paid by Government",
                 "SOCBENKIREG": "Social benefits in kind (D63), received by Government",
                 "SOCBENKIPAG": "Social benefits in kind (D63), paid by Government",
                 "CURTRANSREG": "Other current transfers (D7), received by Government",
                 "CURTRANSPAG": "Other current transfers (D7), received by Government",
                 "INCGDISPG": "Government's gross disposable income (B6G)"}

for reg in CONSGV_r.index.get_level_values("region").unique():
    for key, value in dict_var_consg.items():
        scl_nm = f"sh{key.lower()}"
        var_nm = f"{key}_{reg}"
        val[f"{BASEYEAR}"] = CONSGV_r.loc[(reg)].sum()*scl_govinc.loc[(reg,scl_nm)].sum()
        dict_var_sa[var_nm]  = {"val": val.copy(), "cmt": value, "unit": "million current euros", "idt": "", "eqs": ""}


# Calculation of variables proportional to INCGDISP
dict_var_incg = {"ADJPENREG": "Adjustment for the change in pension entitlements (D8), received by Government",
                 "ADJPENPAG": "Adjustment for the change in pension entitlements (D8), paid by Government",
                 "GSAVG": "Government's gross saving (B8G)",
                 "KTRANSREG": "Capital transfers (D9), received by Government",
                 "KTRANSPAG": "Capital transfers (D9), paid by Government",
                 "FIXCAPPAG": "Government gross capital formation (P5)",
                 "DSTOCKVG": "Changes in inventories (P52) for Government",
                 "DVALUEVG": "Acquisitions less disposals of valuables (P53) for Government", 
                 "ACLDNFNPAG": "Acquisitions less disposals of non-financial non-produced assets (NP), paid by Government",
                 "NLENDBG": "Government's net lending (+) / net borrowing (-)"}

for reg in CONSGV_r.index.get_level_values("region").unique():
    for key, value in dict_var_incg.items():
        scl_nm = f"sh{key.lower()}"
        var_nm = f"{key}_{reg}"
        var_inc = f"INCGDISPG_{reg}"
        val[f"{BASEYEAR}"] = dict_var_sa[var_inc]["val"][f"{BASEYEAR}"]*scl_govinc.loc[(reg,scl_nm)].sum()
        dict_var_sa[var_nm]  = {"val": val.copy(), "cmt": value, "unit": "million current euros", "idt": "", "eqs": ""}
      

# Remaining variables
for reg in GOPSUP_r.index.unique():
    var_nm =  f"GOPSUPG_{reg}"
    val[f"{BASEYEAR}"] = scl_govinc.loc[(reg,"shgopsupg")].sum()*GOPSUP_r.loc[(reg)].sum()
    dict_var_sa[var_nm]  = {"val": val.copy(), "cmt": "Government's gross operating surplus and mixed income (B2G & B3G)", "unit": "million current euros", "idt": "", "eqs": ""}

for reg in DINVV_r.index.unique():
    var_nm =  f"DINVVG_{reg}"
    val[f"{BASEYEAR}"] = scl_govinc.loc[(reg,"shdinvvg")].sum()*DINVV_r.loc[(reg)].sum()
    dict_var_sa[var_nm]  = {"val": val.copy(), "cmt": "Government's gross fixed capital formation", "unit": "million current euros", "idt": "", "eqs": ""}


## CAUTION
# Re-calculation of DSTOCKVG & NLENDBG to ensure consistency 
for reg in DINVV_r.index.unique():
    val[f"{BASEYEAR}"] = dict_var_sa[f"FIXCAPPAG_{reg}"]["val"][f"{BASEYEAR}"]-dict_var_sa[f"DINVVG_{reg}"]["val"][f"{BASEYEAR}"]
    var_nm = f"DSTOCKVG_{reg}"
    dict_var[var_nm]  = {"val": val.copy(), "cmt": "Government's changes in inventories (P52) and acquisitions less disposals of valuables (P53)", "unit": "million current euros", "idt": "", "eqs": ""}
    val[f"{BASEYEAR}"] = dict_var_sa[f"GSAVG_{reg}"]["val"][f"{BASEYEAR}"]+dict_var_sa[f"KTRANSREG_{reg}"]["val"][f"{BASEYEAR}"]-dict_var_sa[f"KTRANSPAG_{reg}"]["val"][f"{BASEYEAR}"]-dict_var_sa[f"DINVVG_{reg}"]["val"][f"{BASEYEAR}"]-dict_var_sa[f"DSTOCKVG_{reg}"]["val"][f"{BASEYEAR}"]-dict_var_sa[f"ACLDNFNPAG_{reg}"]["val"][f"{BASEYEAR}"]
    var_nm = f"NLENDBG_{reg}"
    dict_var_sa[var_nm]  = {"val": val.copy(), "cmt": "Government's net lending (+) / net borrowing (-)", "unit": "million current euros", "idt": "", "eqs": ""}



#### Store variables from sectors' accounts and comments into iode objects

In [29]:
for key in dict_var_sa.keys():
    io.variables[f"{key}"] = np.nan
    io.variables[f"{key}", f"{LOADYEAR}"] = dict_var_sa[f"{key}"]["val"][f"{BASEYEAR}"]
    io.comments[f"{key}"] = dict_var_sa[f"{key}"]["cmt"] + "; Unit:" + dict_var_sa[f"{key}"]["unit"]  
io.variables    

Workspace: Variables
nb variables: 171100
filename: e:\Work\DIAMOND\NeW\data_raw\ghg_emissions\data_ghg.var
sample: 2015Y1:2050Y1
mode: LEVEL

     name    	2015Y1	2016Y1	2017Y1	2018Y1	  2019Y1  	...	2045Y1	2046Y1	2047Y1	2048Y1	2049Y1	2050Y1
ACLDNFNPAG_AU	    na	    na	    na	    na	  -2088.33	...	    na	    na	    na	    na	    na	    na
ACLDNFNPAG_BR	    na	    na	    na	    na	  -1496.79	...	    na	    na	    na	    na	    na	    na
ACLDNFNPAG_CA	    na	    na	    na	    na	      0.00	...	    na	    na	    na	    na	    na	    na
ACLDNFNPAG_CN	    na	    na	    na	    na	-308593.59	...	    na	    na	    na	    na	    na	    na
ACLDNFNPAG_DE	    na	    na	    na	    na	  -1290.78	...	    na	    na	    na	    na	    na	    na
...          	   ...	   ...	   ...	   ...	       ...	...	   ...	   ...	   ...	   ...	   ...	   ...
VAMPV_WM_55  	    na	    na	    na	    na	 163573.78	...	    na	    na	    na	    na	    na	    na
VAMPV_WM_56  	    na	    na	    na	    na	 222696.27	...	    na	 

## Scalars calculation

### Scalars for addressed demands calculation

In [30]:
# Create df for dinvv by demanding region and sector for each supplying sector (aggregated by supplying region) 
mat_gfcf_ss = mat_gfcf.groupby("row_sec").sum()
mat_intcons_ss = exio19.Z.groupby("sector").sum()
mat_enr_ss = exio19.Z.loc[(exio19.Z.index.get_level_values('sector').isin(lst_enr_sect))].groupby("sector").sum()

In [31]:
# INTERMEDIATE CONSUMPTION / ENERGY / INVESTMENT
# Importation of the calculated Investment tables rs*ss/rd*sd

# Calculation of the share of intermediate consumption (without energy) and energy consumption (total intermediate excluded)
dict_scl_io = dict()

# Create df for dinvv by demanding region and sector for each supplying sector (aggregated by supplying region) 
mat_gfcf_ss = mat_gfcf.groupby("row_sec").sum()
mat_intcons_ss = exio19.Z.groupby("sector").sum()
mat_enr_ss = exio19.Z.loc[(exio19.Z.index.get_level_values('sector').isin(lst_enr_sect))].groupby("sector").sum() 
mat_mat_ss = mat_intcons_ss.sub(mat_enr_ss, fill_value=0)
 

# Scalars' calculation
for rd in exio19.Z.index.get_level_values("region").unique():
    for sd in exio19.Z.index.get_level_values("sector").unique():
        for ss in exio19.Z.columns.get_level_values("sector").unique():
            # Intermadiate consumption (excl. energy)
            if dict_var[f"DMATV_{rd}_{sd}"]["val"][f"{BASEYEAR}"] == 0:
                mmat_rd_sd_ss = 1/len(exio19.Z.index.get_level_values("sector").unique()) 
            else:
                mmat_rd_sd_ss = mat_mat_ss.loc[ss,(rd,sd)]/dict_var[f"DMATV_{rd}_{sd}"]["val"][f"{BASEYEAR}"]    
            dict_scl_io[f"mmat_{rd.lower()}_{sd}_{ss}"] = {"val": mmat_rd_sd_ss, "cmt": f"Share of intermediate consumption (except energy) from region {rd} and industry n°{sd} demanded to industry n°{ss}"}
            
            # Gross fixed capital formation
            if dict_var[f"DINVV_{rd}_{sd}"]["val"][f"{BASEYEAR}"] == 0:
                minv_rd_sd_ss = 1/len(lst_enr_sect) 
            else:
                minv_rd_sd_ss = mat_gfcf_ss.loc[ss,(rd,sd)]/mat_gfcf_ss.loc[:,(rd,sd)].sum()
            dict_scl_io[f"minv_{rd.lower()}_{sd}_{ss}"] = {"val": minv_rd_sd_ss, "cmt": f"Share of gross fixed capital formation from region {rd} and industry n°{sd} demanded to industry n°{ss}"}    
        
        for ss in lst_enr_sect:
            # Energy consumption
            if dict_var[f"DENRTOTV_{rd}_{sd}"]["val"][f"{BASEYEAR}"] == 0:
                menr_rd_sd_ss = 1/len(exio19.Z.index.get_level_values("sector").unique())  
            else:
                menr_rd_sd_ss = mat_enr_ss.loc[ss,(rd,sd)]/dict_var[f"DENRTOTV_{rd}_{sd}"]["val"][f"{BASEYEAR}"]  
            dict_scl_io[f"menr_{rd.lower()}_{sd}_{ss}"] = {"val": menr_rd_sd_ss, "cmt": f"Share of energy consumption from region {rd} and industry n°{sd} demanded to industry n°{ss}"}
       

In [32]:
# NPISH & GOVERNMENT
# Sum the supplying sectors
consnp_agg = exio19.Y.loc[:,(exio19.Y.columns.get_level_values("category") == "CONSNPV")].groupby(level=1).sum()
consg_agg = exio19.Y.loc[:,(exio19.Y.columns.get_level_values("category") == "CONSGV")].groupby(level=1).sum()

# Calculation of the share of each supplyoing sector in the total
for rd in exio19.Z.index.get_level_values("region").unique():
    for ss in exio19.Z.columns.get_level_values("sector").unique():
        # CONSNPV
        mconsnp_rd_ss = consnp_agg.loc[ss,rd]/consnp_agg.copy().sum().loc[rd]
        val = mconsnp_rd_ss.values[0]
        dict_scl_io[f"mconsnp_{rd.lower()}_{ss}"] = {"val": val, "cmt": f"Share of NPISH final consumption from region {rd} demanded to industry n°{ss}"}
        # CONSGV
        mconsg_rd_ss = consg_agg.loc[ss,rd]/consg_agg.copy().sum().loc[rd]
        val = mconsg_rd_ss.values[0]
        dict_scl_io[f"mconsg_{rd.lower()}_{ss}"] = {"val": val, "cmt": f"Share of Government' final consumption from region {rd} demanded to industry n°{ss}"}
      

In [33]:
### Calculation scalars for adressed demands for households'

# Consumption functions linked to economic sector (no aggregates)
# Cases where one sector maps with several consumption functions
dict_tmp = dict()
# 03
dict_splitted_cons = {"03": ["HTCL","EQUP"],
            "14": ["HTCL","ICEF","PHEF"],
            "31": ["ICEV","PHEV","BELV"],
            "32": ["ICEV","PHEV","BELV"],
            "35": ["HTCL","PHEF", "BELF"],
            "36": ["HTCL","PHEF", "BELF"],
            "37": ["HTCL","PHEF", "BELF"],
            "38": ["HTCL","ICEF"]}

dict_splitted_cons_enr = {"03": ["HSBM","EQUP"],
                        "07": ["HGAS", "HBGS"],
            "14": ["HOIL","IFOI","IFLB","PFOI","PFBF"],
            "31": ["ICEV","PHEV","BELV"],
            "32": ["ICEV","PHEV","BELV"],
            "35": ["HELC","PFEL","BELF"],
            "36": ["HELC","PFEL","BELF"],
            "37": ["HELC","PFEL","BELF"],
            "38": ["HGAS","HBGS","IFGS","IFBG"],
            "39": ["HHEA","HGEO","HSLT"]}

for r in exio19.Y.index.get_level_values("region").unique(): 
    # for (key, items) in dict_splitted_cons.items():
    for (key, items) in dict_splitted_cons_enr.items():
        for cs in items:
            if dict_var[f"CONSHV_{r}_{cs}"]["val"][f"{BASEYEAR}"] == 0:
                val = 1/len(items)
            else:
                val = dict_var[f"CONSHV_{r}_{key}"]["val"][f"{BASEYEAR}"]*param_split[f"sh_{cs.lower()}_{r}_{key}"]/dict_var[f"CONSHV_{r}_{cs}"]["val"][f"{BASEYEAR}"] 
            dict_scl_io[f"mconsh_{r.lower()}_{cs.lower()}_{key}"] =  {"val": val, "cmt": f"Share in households consumption from region {r} demanded to industry n°{key}"}



# Consumption functions linked to economic sector (no aggregates),
# Caution only functions with a unique correspondance with sector
dict_cons_add = {"FOOD": [["01","02","04","09"],["Food"]], 
                    "CLOT": [["10"],["Clothes"]], 
                    "RENT": [["52"],["Rent"]], 
                    "WAWA": [["34", "40"],["Wastes & water"]], 
                    "HTCL": [["05","06","07","13","39"], ["Heating and cooling"]],
                    "ICEF": [["38"], "Internal combustion engine fuels"],
                    "RAIL": [["44"],["Railways"]],
                    "OINL": [["45"],["Other inland transport"]],
                    "AIRT": [["48"],["Air transports"]],
                    "WATT": [["46","47"],["Water transports"]], 
                    "OTTR": [["49"],["Other transports"]],
                    "EDUC": [["57"],["Education"]],
                    "HLTH": [["58"],["Health"]],
                    "ONMS": [["56","59"],["Other non-market services"]],
                    "HRES": [["43"],["Hotels & restaurants"]],
                    "INFC": [["53"],["Information & communication"]],
                    "OLEI": [["12"],["Other leisure"]],
                    "EQUP": [["11","15","16","17","18","19","20","21","22","23","24","25","26","27","28","29","30","33","41"],["Equipments"]],
                    "OOTH": [["08","50","51","54","55"],["Others"]],
                    "TRAD": [["42"],["Trade"]]}

dict_cons_add_enr = {"FOOD": [["01","02","04","09"],["Food"]], 
                    "CLOT": [["10"],["Clothes"]], 
                    "RENT": [["52"],["Rent"]], 
                    "WAWA": [["34", "40"],["Wastes & water"]],
                    "HCOM": [["05", "13"],["Heating and Cooling - Solid fossil fuels"]],
                    "HOIL": [["06"],["Heating and Cooling - Liquid fossil fuels"]],
                    "RAIL": [["44"],["Railways"]],
                    "OINL": [["45"],["Other inland transport"]],
                    "AIRT": [["48"],["Air transports"]],
                    "WATT": [["46","47"],["Water transports"]], 
                    "OTTR": [["49"],["Other transports"]],
                    "EDUC": [["57"],["Education"]],
                    "HLTH": [["58"],["Health"]],
                    "ONMS": [["56","59"],["Other non-market services"]],
                    "HRES": [["43"],["Hotels & restaurants"]],
                    "INFC": [["53"],["Information & communication"]],
                    "OLEI": [["12"],["Other leisure"]],
                    "EQUP": [["11","15","16","17","18","19","20","21","22","23","24","25","26","27","28","29","30","33","41"],["Equipments"]],
                    "OOTH": [["08","50","51","54","55"],["Others"]],
                    "TRAD": [["42"],["Trade"]]}

for rd in exio19.Y.index.get_level_values("region").unique():
    # for (key, items) in dict_cons_add.items():
    for (key, items) in dict_cons_add_enr.items():
        ss = items[0]
        nm = items[1]
        value = 0
        for item in ss:
            mconsh_rd_key_ss = dict_var[f"CONSHV_{rd}_{item}"]["val"][f"{BASEYEAR}"]/dict_var[f"CONSHV_{rd}_{key}"]["val"][f"{BASEYEAR}"]
            dict_scl_io[f"mconsh_{rd.lower()}_{key.lower()}_{item}"] = {"val": mconsh_rd_key_ss, "cmt": f"Share of {nm[0]} in households consumption from region {rd} demanded to industry n°{ss}"}
            

#### Store scalars and comments into iode objects

In [34]:
for key in dict_scl_io.keys():
    io.scalars[f"{key}"] = dict_scl_io[f"{key}"]["val"]
    io.comments[f"{key}"] = dict_scl_io[f"{key}"]["cmt"]
io.scalars    


Workspace: Scalars
nb scalars: 170518
filename: e:\Work\DIAMOND\NeW\data_raw\ghg_emissions\scl_ghg.scl

     name    	value 	relax 	std
emch4fc_comb 	0.0010	1.0000	 na
emch4fc_gas  	0.0010	1.0000	 na
emch4fc_oil  	0.0030	1.0000	 na
emch4fe_comb 	0.7500	1.0000	 na
emch4fe_gas  	1.8000	1.0000	 na
...          	   ...	   ...	...
mmat_wm_59_55	0.1546	1.0000	 na
mmat_wm_59_56	0.0167	1.0000	 na
mmat_wm_59_57	0.0018	1.0000	 na
mmat_wm_59_58	0.0012	1.0000	 na
mmat_wm_59_59	0.0964	1.0000	 na

##### Verifying sum over supplying sectors = 1

In [35]:
tmp = 0
tmp1 = 0
tmp2 = 0 
for ss in exio19.Z.columns.get_level_values("sector").unique():
    tmp = tmp + io.scalars[f"minv_fr_03_{ss}"].value
    tmp1 = tmp1 + io.scalars[f"mmat_fr_03_{ss}"].value
for ss in lst_enr_sect:    
    tmp2 = tmp2 + io.scalars[f"menr_fr_03_{ss}"].value
print(tmp, tmp1, tmp2)

1.0000000000000002 1.0 0.9999999999999999


In [36]:
path_save = pathwork + "Data/"
io.variables.save(f"{path_save}rawdata_v1.00.var")
# io.identities.save(f"{path_save}rawdata_v1.00.idt")
# io.equations.save(f"{path_save}rawdata_v0.94.eqs")
io.scalars.save(f"{path_save}rawdata_v1.00.scl")
io.comments.save(f"{path_save}rawdata_v1.00.cmt")

Saving E:/Work/DIAMOND/NeW/Data/rawdata_v1.00.var
171100 objects saved
Saving E:/Work/DIAMOND/NeW/Data/rawdata_v1.00.scl
170518 objects saved
Saving E:/Work/DIAMOND/NeW/Data/rawdata_v1.00.cmt
320124 objects saved


In [ ]:
pathoth = pathwork + f"Data_Raw/Others"

# Population data
pathpop = pathwork + "Others/Pop/"
file_pop_nm = "pop_ssp2.var"
ld_pop = pathpop + file_pop_nm


# Unemployment rates data
pathundata = pathwork + "Others/Unemp/"
file_un_nm = "unempra.var"
ld_un = pathundata + file_un_nm
io.variables.merge_from(ld_un)


# Interest rate data
fl_nm_inr = "OECD_Long_term_Interest_rates.xlsx"
intr = pd.read_excel(os.path.join(pathoth ,fl_nm_inr), sheet_name="rzrl", index_col="vars")
io.variables.from_frame(intr)

# Inflation rate data
fl_nm_infl = "OECD_Inflation_rates.xlsx"
infl = pd.read_excel(os.path.join(pathoth, fl_nm_infl), sheet_name="inflr", index_col="vars")
io.variables.from_frame(infl)

SMP = SMPSTRY + ":" + SMPENDY
io.variables.sample = f'{SMP}'

# Retropolate values (same as 2019 for 2015 to 2018)
for var in io.variables.names:
    io.identities[f"{var}"] = f"{var}[{BASEYEAR}]"
    io.identities.execute(f"{var}", f"{SMPSTRY}", "2018Y1")
io.identities.clear    

# Extrapolate values (constant) (except pop)
io.variables.extrapolate(SimulationInitialization.TM1_A, "2020Y1", f"{SMPENDY}")

# Popuiation data
pathpop = pathwork + "Data_Raw/Others/Pop/"
file_pop_nm = "pop_ssp2.var"
ld_pop = pathpop + file_pop_nm
io.variables.merge_from(ld_pop)

# Save variables
sv_vars = path_save + "Rawdata_v1.00_extrapoled.var"
io.variables.save(sv_vars)